# Additional requested analyses

Standalone supplementary analyses to address requested sensitivity and clarification analyses about overall and within-stratum association, trivial direction-only shortcuts, and wording that is more accurate than "approximate."


## 1. Setup and data loading


In [ ]:
from __future__ import annotations

from datetime import date
from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from matplotlib.transforms import blended_transform_factory
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import confusion_matrix
from statsmodels.stats.inter_rater import cohens_kappa

try:
    import scienceplots  # noqa: F401

    plt.style.use(["science", "no-latex"])
    STYLE_NAME = "science"
except Exception:
    plt.style.use("default")
    STYLE_NAME = "default"

mpl.rcParams.update(
    {
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.linestyle": "--",
        "grid.linewidth": 0.4,
        "grid.alpha": 0.35,
        "axes.unicode_minus": False,
        "savefig.bbox": "tight",
    }
)

WORKING_DIR = Path.cwd()
RESULTS_ROOT = WORKING_DIR / "results"

OUTPUT_DIR = RESULTS_ROOT / date.today().strftime("%Y-%m-%d")
FOLLOWUP_DIR = OUTPUT_DIR / "additional_requested_analyses"
FOLLOWUP_DIR.mkdir(parents=True, exist_ok=True)

WORKBOOK_CANDIDATES = [
    WORKING_DIR / "NNT_LRs_08-26-2025.xlsx",
    WORKING_DIR / "nnt_lrs_with_estimated.xlsx",
]
WORKBOOK_PATH = next((path for path in WORKBOOK_CANDIDATES if path.exists()), None)
if WORKBOOK_PATH is None:
    tried = "\n".join(f"- {path}" for path in WORKBOOK_CANDIDATES)
    raise FileNotFoundError(
        "Could not find the source workbook. Tried:\n"
        f"{tried}"
    )

SHEET_NAME = "Master"
header_only = pd.read_excel(WORKBOOK_PATH, sheet_name=SHEET_NAME, nrows=0)
AVAILABLE_COLUMNS = header_only.columns.tolist()

REQUIRED_COLUMNS = [
    "lr_reported",
    "lr_gpt-4o-2024-11-20",
    "lr_o3-2025-04-16",
    "lr_gpt-5",
    "Feature Type",
]
missing_required = [col for col in REQUIRED_COLUMNS if col not in AVAILABLE_COLUMNS]
if missing_required:
    available = ", ".join(map(str, AVAILABLE_COLUMNS))
    missing = ", ".join(missing_required)
    raise KeyError(
        "The workbook is missing required columns: "
        f"{missing}. Available columns are: {available}"
    )


def find_optional_column(
    columns,
    exact_names=(),
    contains=(),
    exclude=(),
):
    exclude_lower = {str(col).strip().lower() for col in exclude if col is not None}
    exact_lookup = {str(col).strip().lower(): col for col in columns}

    for candidate in exact_names:
        match = exact_lookup.get(candidate.strip().lower())
        if match is not None and str(match).strip().lower() not in exclude_lower:
            return match

    for col in columns:
        low = str(col).strip().lower()
        if low in exclude_lower:
            continue
        if any(token in low for token in contains):
            return col
    return None


FINDING_COLUMN = find_optional_column(
    AVAILABLE_COLUMNS,
    exact_names=("finding",),
    contains=("finding",),
)
CONDITION_COLUMN = find_optional_column(
    AVAILABLE_COLUMNS,
    exact_names=("condition",),
    contains=("condition", "diagnosis", "disease"),
    exclude=(FINDING_COLUMN,),
)
DIRECTION_COLUMN = find_optional_column(
    AVAILABLE_COLUMNS,
    contains=(
        "direction",
        "polarity",
        "present_absent",
        "present/absent",
        "positive_negative",
        "positive/negative",
        "finding direction",
        "result direction",
    ),
    exclude=tuple(REQUIRED_COLUMNS) + (FINDING_COLUMN, CONDITION_COLUMN),
)

if FINDING_COLUMN is None:
    available = ", ".join(map(str, AVAILABLE_COLUMNS))
    raise KeyError(
        "The workbook is missing the finding column required to reconstruct true LR+ and LR− labels. "
        f"Available columns are: {available}"
    )

SCRAPE_PREFIX_FIXES = {
    "Patient  has:": "Patient has:",
    "Patient does has:": "Patient does not have:",
}
SOURCE_LABEL_ORDER = ["LR+", "LR−"]


def normalize_scrape_prefixes(series: pd.Series) -> pd.Series:
    normalized = (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    for old, new in SCRAPE_PREFIX_FIXES.items():
        normalized = normalized.str.replace(old, new, regex=False)
    return normalized


df = pd.read_excel(WORKBOOK_PATH, sheet_name=SHEET_NAME)

LR_COLUMNS = [
    "lr_reported",
    "lr_gpt-4o-2024-11-20",
    "lr_o3-2025-04-16",
    "lr_gpt-5",
]


def coerce_positive_numeric(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.where((numeric > 0) & np.isfinite(numeric))


coercion_rows = []
for col in LR_COLUMNS:
    raw = df[col].copy()
    numeric = pd.to_numeric(raw, errors="coerce")
    positive = numeric.where((numeric > 0) & np.isfinite(numeric))
    coercion_rows.append(
        {
            "column": col,
            "rows": int(len(df)),
            "nonmissing_raw": int(raw.notna().sum()),
            "numeric_after_coercion": int(numeric.notna().sum()),
            "positive_after_filtering": int(positive.notna().sum()),
            "dropped_nonpositive_or_nonfinite": int(
                numeric.notna().sum() - positive.notna().sum()
            ),
        }
    )
    df[col] = positive
    df[f"ln_{col}"] = np.log(df[col])

df[FINDING_COLUMN] = df[FINDING_COLUMN].fillna("").astype(str)
df["finding_source_normalized"] = normalize_scrape_prefixes(df[FINDING_COLUMN])
df["source_lr_label"] = np.select(
    [
        df["finding_source_normalized"].str.startswith("Patient has:"),
        df["finding_source_normalized"].str.startswith("Patient does not have:"),
    ],
    SOURCE_LABEL_ORDER,
    default="",
)

unclassified_source_mask = df["source_lr_label"].eq("")
if unclassified_source_mask.any():
    unclassified_columns = [
        col for col in [FINDING_COLUMN, CONDITION_COLUMN, "lr_reported"] if col is not None
    ]
    unclassified_path = FOLLOWUP_DIR / "QC - unclassified source lr labels.csv"
    df.loc[unclassified_source_mask, unclassified_columns].to_csv(unclassified_path, index=False)
    raise ValueError(
        "Could not classify all rows into true LR+ or LR− labels from the scrape prefixes. "
        f"Problem rows were written to {unclassified_path}."
    )

source_label_counts = (
    df["source_lr_label"]
    .value_counts()
    .reindex(SOURCE_LABEL_ORDER, fill_value=0)
    .rename_axis("source_lr_label")
    .reset_index(name="n")
)

df["Feature Type"] = (
    df["Feature Type"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .replace({"": "Missing"})
)
df["reported_stratum"] = np.select(
    [df["lr_reported"] < 1, df["lr_reported"] > 1],
    ["LR < 1", "LR > 1"],
    default="LR = 1",
)

MODEL_SPECS = [
    {
        "label": "GPT‑4o",
        "kind": "model",
        "lr_col": "lr_gpt-4o-2024-11-20",
        "ln_col": "ln_lr_gpt-4o-2024-11-20",
        "color": "#c23b31",
    },
    {
        "label": "o3",
        "kind": "model",
        "lr_col": "lr_o3-2025-04-16",
        "ln_col": "ln_lr_o3-2025-04-16",
        "color": "#c89211",
    },
    {
        "label": "GPT‑5",
        "kind": "model",
        "lr_col": "lr_gpt-5",
        "ln_col": "ln_lr_gpt-5",
        "color": "#2f8f58",
    },
]

OPTIONAL_COLUMNS = {
    "finding": FINDING_COLUMN,
    "condition": CONDITION_COLUMN,
    "direction": DIRECTION_COLUMN,
}
OPTIONAL_COLUMNS = {key: value for key, value in OPTIONAL_COLUMNS.items() if value is not None}

print(f"Notebook style: {STYLE_NAME}")
print(f"Working directory: {WORKING_DIR}")
print(f"Workbook: {WORKBOOK_PATH.name}")
print(f"Sheet: {SHEET_NAME}")
print(f"Output directory: {FOLLOWUP_DIR}")
print(f"Optional contextual columns found: {OPTIONAL_COLUMNS if OPTIONAL_COLUMNS else 'none'}")
print(
    "Reconstructed source LR labels from scrape prefixes after normalizing two known prefix artifacts: "
    "`Patient  has:` and `Patient does has:`."
)

coercion_summary = pd.DataFrame(coercion_rows)
display(coercion_summary)
display(source_label_counts)

preview_columns = REQUIRED_COLUMNS + [col for col in OPTIONAL_COLUMNS.values() if col not in REQUIRED_COLUMNS]
preview_columns += ["source_lr_label"]
preview_columns += [f"ln_{col}" for col in LR_COLUMNS]
preview_columns = [col for col in preview_columns if col in df.columns]
display(df[preview_columns].head())


## 2. Shared helper functions


In [ ]:
QUAL_LABELS = [
    "Strong Negative",
    "Moderate Negative",
    "Weak Negative",
    "Negligible",
    "Weak Positive",
    "Moderate Positive",
    "Strong Positive",
]

METHOD_ORDER = ["GPT‑4o", "o3", "GPT‑5", "LR+/LR− label-only heuristic", "Sign-only heuristic", "Sign + feature-type heuristic"]
CONST_TOL = 1e-12


def bin_qualitative_lr(lr_series: pd.Series) -> pd.Categorical:
    lr = coerce_positive_numeric(lr_series)
    conditions = [
        lr <= 0.10,
        (lr > 0.10) & (lr <= 0.20),
        (lr > 0.20) & (lr < 0.50),
        (lr >= 0.50) & (lr < 2.00),
        (lr >= 2.00) & (lr < 5.00),
        (lr >= 5.00) & (lr < 10.00),
        lr >= 10.00,
    ]
    labels = np.select(conditions, QUAL_LABELS, default=None)
    return pd.Categorical(labels, categories=QUAL_LABELS, ordered=True)


def paired_xy(
    frame: pd.DataFrame,
    x_col: str,
    y_col: str,
    mask=None,
) -> tuple[np.ndarray, np.ndarray]:
    if mask is None:
        mask = pd.Series(True, index=frame.index)
    mask = pd.Series(mask, index=frame.index).fillna(False)
    pair_mask = mask & frame[x_col].notna() & frame[y_col].notna()
    x = frame.loc[pair_mask, x_col].to_numpy(dtype=float)
    y = frame.loc[pair_mask, y_col].to_numpy(dtype=float)
    return x, y


def geometric_mean(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0)]
    if values.size == 0:
        return np.nan
    return float(np.exp(np.mean(np.log(values))))


def leave_one_out_geometric_mean(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    out = np.full(values.shape, np.nan, dtype=float)
    valid = np.isfinite(values) & (values > 0)
    if valid.sum() <= 1:
        return out
    logs = np.log(values[valid])
    loo_logs = (logs.sum() - logs) / (len(logs) - 1)
    out[np.where(valid)[0]] = np.exp(loo_logs)
    return out


def ols_calibration(x: np.ndarray, y: np.ndarray) -> dict:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_sd = float(np.nanstd(x, ddof=1)) if x.size > 1 else np.nan
    y_sd = float(np.nanstd(y, ddof=1)) if y.size > 1 else np.nan
    if x.size < 3 or not np.isfinite(x_sd) or x_sd <= CONST_TOL:
        return {
            "intercept": np.nan,
            "intercept_ci_low": np.nan,
            "intercept_ci_high": np.nan,
            "slope": np.nan,
            "slope_ci_low": np.nan,
            "slope_ci_high": np.nan,
            "r2": np.nan,
        }

    X = sm.add_constant(x, has_constant="add")
    fit = sm.OLS(y, X, missing="drop").fit()
    ci = fit.conf_int()

    return {
        "intercept": float(fit.params[0]),
        "intercept_ci_low": float(ci[0, 0]),
        "intercept_ci_high": float(ci[0, 1]),
        "slope": float(fit.params[1]),
        "slope_ci_low": float(ci[1, 0]),
        "slope_ci_high": float(ci[1, 1]),
        "r2": 0.0 if (np.isfinite(y_sd) and y_sd <= CONST_TOL) else float(fit.rsquared),
    }


def pearson_stats(x: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size < 3:
        return np.nan, np.nan
    x_sd = float(np.nanstd(x, ddof=1))
    y_sd = float(np.nanstd(y, ddof=1))
    if (not np.isfinite(x_sd)) or (not np.isfinite(y_sd)) or x_sd <= CONST_TOL or y_sd <= CONST_TOL:
        return 0.0, np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = stats.pearsonr(x, y)
    if hasattr(result, "statistic"):
        return float(result.statistic), float(result.pvalue)
    return float(result[0]), float(result[1])


def spearman_stats(x: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size < 3:
        return np.nan, np.nan
    x_sd = float(np.nanstd(x, ddof=1))
    y_sd = float(np.nanstd(y, ddof=1))
    if (not np.isfinite(x_sd)) or (not np.isfinite(y_sd)) or x_sd <= CONST_TOL or y_sd <= CONST_TOL:
        return 0.0, np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = stats.spearmanr(x, y, nan_policy="omit")
    if hasattr(result, "statistic"):
        return float(result.statistic), float(result.pvalue)
    return float(result[0]), float(result[1])


def lins_ccc(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_sd = float(np.nanstd(x, ddof=1)) if x.size > 1 else np.nan
    y_sd = float(np.nanstd(y, ddof=1)) if y.size > 1 else np.nan
    if x.size < 2 or not np.isfinite(x_sd) or not np.isfinite(y_sd) or x_sd <= CONST_TOL or y_sd <= CONST_TOL:
        return np.nan
    mean_x = float(np.mean(x))
    mean_y = float(np.mean(y))
    var_x = float(np.var(x, ddof=1))
    var_y = float(np.var(y, ddof=1))
    cov_xy = float(np.cov(x, y, ddof=1)[0, 1])
    return float((2 * cov_xy) / (var_x + var_y + (mean_x - mean_y) ** 2))


def error_metrics_log_scale(x: np.ndarray, y: np.ndarray) -> dict:
    errors = np.asarray(y, dtype=float) - np.asarray(x, dtype=float)
    abs_errors = np.abs(errors)
    rmse_log = float(np.sqrt(np.mean(errors**2))) if errors.size else np.nan
    return {
        "mean_log_error": float(np.mean(errors)) if errors.size else np.nan,
        "median_log_error": float(np.median(errors)) if errors.size else np.nan,
        "mean_abs_log_error": float(np.mean(abs_errors)) if abs_errors.size else np.nan,
        "median_abs_log_error": float(np.median(abs_errors)) if abs_errors.size else np.nan,
        "rmse_log": rmse_log,
        "typical_error_factor": float(np.exp(np.median(abs_errors))) if abs_errors.size else np.nan,
        "rmse_error_factor": float(np.exp(rmse_log)) if np.isfinite(rmse_log) else np.nan,
    }


def bland_altman_ratio_metrics(x: np.ndarray, y: np.ndarray) -> dict:
    ln_ref = np.asarray(x, dtype=float)
    ln_model = np.asarray(y, dtype=float)
    ln_ratio = ln_ref - ln_model
    if ln_ratio.size == 0:
        return {
            "mean_bias_xfold": np.nan,
            "loa95_lower_xfold": np.nan,
            "loa95_upper_xfold": np.nan,
        }

    mean_log_ratio = float(np.mean(ln_ratio))
    sd_log_ratio = float(np.std(ln_ratio, ddof=1)) if ln_ratio.size > 1 else np.nan
    if np.isfinite(sd_log_ratio):
        loa_lower_log = mean_log_ratio - 1.96 * sd_log_ratio
        loa_upper_log = mean_log_ratio + 1.96 * sd_log_ratio
    else:
        loa_lower_log = np.nan
        loa_upper_log = np.nan

    return {
        "mean_bias_xfold": float(np.exp(mean_log_ratio)),
        "loa95_lower_xfold": float(np.exp(loa_lower_log)) if np.isfinite(loa_lower_log) else np.nan,
        "loa95_upper_xfold": float(np.exp(loa_upper_log)) if np.isfinite(loa_upper_log) else np.nan,
    }


def bland_altman_xy(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    ln_ref = np.asarray(x, dtype=float)
    ln_model = np.asarray(y, dtype=float)
    return (ln_ref + ln_model) / 2.0, ln_ref - ln_model


def bland_altman_axis_limits(
    pairs,
    pad_x: float = 0.05,
    pad_y: float = 0.05,
) -> tuple[tuple[float, float], tuple[float, float]]:
    means_all = []
    diffs_all = []
    for x, y in pairs:
        if len(x) == 0 or len(y) == 0:
            continue
        means, diffs = bland_altman_xy(x, y)
        means_all.append(means)
        diffs_all.append(diffs)

    if not means_all or not diffs_all:
        return (-5.0, 5.0), (-3.0, 3.0)

    means_combined = np.concatenate(means_all)
    diffs_combined = np.concatenate(diffs_all)

    x_low = float(np.nanmin(means_combined))
    x_high = float(np.nanmax(means_combined))
    if not np.isfinite(x_low) or not np.isfinite(x_high) or x_low == x_high:
        x_limits = (-5.0, 5.0)
    else:
        x_span = x_high - x_low
        x_pad = x_span * pad_x
        x_limits = (x_low - x_pad, x_high + x_pad)

    y_extent = float(np.nanmax(np.abs(diffs_combined))) if diffs_combined.size else np.nan
    if not np.isfinite(y_extent) or y_extent <= 0:
        y_limits = (-3.0, 3.0)
    else:
        y_extent = y_extent * (1 + pad_y)
        y_limits = (-y_extent, y_extent)

    return x_limits, y_limits


def format_lr_tick(value: float, _position: float) -> str:
    lr_value = float(np.exp(value))
    if not np.isfinite(lr_value):
        return ""
    if lr_value >= 10 or lr_value < 0.1:
        return f"{lr_value:.2g}"
    return f"{lr_value:.1f}"


def plot_bland_altman_panel(
    ax,
    x: np.ndarray,
    y: np.ndarray,
    *,
    color: str,
    x_limits: tuple[float, float],
    y_limits: tuple[float, float],
) -> None:
    if len(x) < 2:
        ax.text(0.5, 0.5, "Insufficient paired rows", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return

    means, diffs = bland_altman_xy(x, y)
    stats = bland_altman_ratio_metrics(x, y)
    mean_log_ratio = float(np.mean(diffs))
    sd_log_ratio = float(np.std(diffs, ddof=1)) if len(diffs) > 1 else np.nan
    loa_lower_log = mean_log_ratio - 1.96 * sd_log_ratio if np.isfinite(sd_log_ratio) else np.nan
    loa_upper_log = mean_log_ratio + 1.96 * sd_log_ratio if np.isfinite(sd_log_ratio) else np.nan

    ax.scatter(means, diffs, s=18, alpha=0.35, color=color, edgecolor="none", zorder=2)
    ax.axhline(mean_log_ratio, color="#1f1f1f", linewidth=1.2, linestyle="-", zorder=3)
    if np.isfinite(loa_lower_log):
        ax.axhline(loa_lower_log, color="#1f1f1f", linewidth=1.0, linestyle="--", zorder=3)
    if np.isfinite(loa_upper_log):
        ax.axhline(loa_upper_log, color="#1f1f1f", linewidth=1.0, linestyle="--", zorder=3)

    trans = blended_transform_factory(ax.transAxes, ax.transData)
    ax.text(
        0.98,
        mean_log_ratio,
        f"Mean bias:\n{stats['mean_bias_xfold']:.2f}x",
        transform=trans,
        ha="right",
        va="center",
        fontsize=9,
        bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none", "pad": 0.2},
    )
    if np.isfinite(loa_upper_log):
        ax.text(
            0.98,
            loa_upper_log,
            f"95% LoA: {stats['loa95_upper_xfold']:.2f}x",
            transform=trans,
            ha="right",
            va="bottom",
            fontsize=8.5,
        )
    if np.isfinite(loa_lower_log):
        ax.text(
            0.98,
            loa_lower_log,
            f"95% LoA: {stats['loa95_lower_xfold']:.2f}x",
            transform=trans,
            ha="right",
            va="top",
            fontsize=8.5,
        )

    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    tick_formatter = FuncFormatter(format_lr_tick)
    ax.xaxis.set_major_formatter(tick_formatter)
    ax.yaxis.set_major_formatter(tick_formatter)
    ax.set_xlabel("Geometric Mean LR")
    ax.set_ylabel("LR ratio (Reported / Model)")


def summarize_method(
    frame: pd.DataFrame,
    method_spec: dict,
    stratum: str = "Overall",
    mask=None,
) -> dict:
    x, y = paired_xy(frame, "ln_lr_reported", method_spec["ln_col"], mask=mask)
    calibration = ols_calibration(x, y)
    pearson_r, pearson_p = pearson_stats(x, y)
    spearman_rho, spearman_p = spearman_stats(x, y)
    errors = error_metrics_log_scale(x, y)
    bland_altman = bland_altman_ratio_metrics(x, y)
    return {
        "method": method_spec["label"],
        "kind": method_spec["kind"],
        "prediction_variant": method_spec.get("variant", "primary"),
        "stratum": stratum,
        "n": int(len(x)),
        **calibration,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
        **errors,
        "mean_bias_xfold": bland_altman["mean_bias_xfold"],
        "loa95_lower_xfold": bland_altman["loa95_lower_xfold"],
        "loa95_upper_xfold": bland_altman["loa95_upper_xfold"],
        "ccc_log": lins_ccc(x, y),
    }


def shared_axis_limits(pairs, pad: float = 0.05) -> tuple[float, float]:
    arrays = []
    for x, y in pairs:
        if len(x) == 0 or len(y) == 0:
            continue
        arrays.extend([np.asarray(x, dtype=float), np.asarray(y, dtype=float)])
    if not arrays:
        return (-5.0, 5.0)
    combined = np.concatenate(arrays)
    low = float(np.nanmin(combined))
    high = float(np.nanmax(combined))
    if not np.isfinite(low) or not np.isfinite(high) or low == high:
        return (-5.0, 5.0)
    span = high - low
    padding = span * pad
    return low - padding, high + padding


def quantile_bin_means(x: np.ndarray, y: np.ndarray, n_bins: int = 8) -> tuple[np.ndarray, np.ndarray]:
    if len(x) < 4:
        return np.array([]), np.array([])
    q = min(max(4, n_bins), len(np.unique(x)))
    try:
        bins = pd.qcut(x, q=q, duplicates="drop")
    except ValueError:
        bins = pd.cut(x, bins=q)
    grouped = (
        pd.DataFrame({"x": x, "y": y, "bin": bins})
        .dropna()
        .groupby("bin", observed=True)
        .mean(numeric_only=True)
    )
    return grouped["x"].to_numpy(), grouped["y"].to_numpy()


def calibration_legend_handles(color: str):
    return [
        Line2D([0], [0], marker="o", linestyle="", markersize=5, color=color, alpha=0.35, label="Pairs"),
        Line2D([0], [0], linestyle="--", linewidth=1.0, color="#666666", label="Identity"),
        Line2D([0], [0], linestyle="-", linewidth=1.4, color="#1f1f1f", label="OLS"),
        Line2D([0], [0], marker="o", linewidth=1.8, markersize=4, color=color, label="Binned means"),
        Line2D([0], [0], linestyle=":", linewidth=1.4, color=color, label="LOWESS"),
        Line2D([0], [0], linestyle="-.", linewidth=1.2, color="#333333", label="Isotonic"),
    ]


def plot_calibration_panel(
    ax,
    x: np.ndarray,
    y: np.ndarray,
    *,
    title: str,
    color: str,
    limits: tuple[float, float],
    annotate: bool = True,
) -> None:
    if len(x) < 3:
        ax.text(0.5, 0.5, "Insufficient paired rows", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return

    metrics = summarize_method(
        pd.DataFrame({"ln_lr_reported": x, "__tmp__": y}),
        {"label": title, "kind": "tmp", "ln_col": "__tmp__"},
        mask=pd.Series(True, index=np.arange(len(x))),
    )

    line_x = np.linspace(limits[0], limits[1], 200)
    calibration = ols_calibration(x, y)

    ax.scatter(x, y, s=18, alpha=0.35, color=color, edgecolor="none", zorder=2)
    ax.plot(line_x, line_x, linestyle="--", linewidth=1.0, color="#666666", zorder=1)
    ax.plot(
        line_x,
        calibration["intercept"] + calibration["slope"] * line_x,
        color="#1f1f1f",
        linewidth=1.4,
        zorder=3,
    )

    bx, by = quantile_bin_means(x, y, n_bins=8)
    if len(bx):
        ax.plot(bx, by, marker="o", markersize=4.5, linewidth=1.8, color=color, zorder=4)

    if len(x) >= 20:
        smooth = lowess(y, x, frac=0.35, return_sorted=True)
        ax.plot(smooth[:, 0], smooth[:, 1], linestyle=":", linewidth=1.4, color=color, zorder=5)

    if len(np.unique(x)) >= 5:
        iso = IsotonicRegression(out_of_bounds="clip")
        y_iso = iso.fit_transform(x, y)
        order = np.argsort(x)
        ax.plot(x[order], y_iso[order], linestyle="-.", linewidth=1.2, color="#333333", zorder=6)

    if annotate:
        annotation = (
            f"n = {len(x)}\n"
            f"slope = {metrics['slope']:.2f}\n"
            f"Pearson r = {metrics['pearson_r']:.2f}"
        )
        ax.text(
            0.03,
            0.97,
            annotation,
            ha="left",
            va="top",
            transform=ax.transAxes,
            fontsize=8.5,
            bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "#cccccc"},
        )

    ax.set_title(title)
    ax.set_xlim(*limits)
    ax.set_ylim(*limits)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Literature-reported ln(LR)")
    ax.set_ylabel("LLM-generated ln(LR)")


def sort_methods(frame: pd.DataFrame) -> pd.DataFrame:
    ordered = frame.copy()
    ordered["method"] = pd.Categorical(ordered["method"], categories=METHOD_ORDER, ordered=True)
    return ordered.sort_values([col for col in ["stratum", "method"] if col in ordered.columns]).reset_index(drop=True)


def add_sign_only_baseline_constant(frame: pd.DataFrame, lr_col: str, out_col: str) -> None:
    predictions = pd.Series(np.nan, index=frame.index, dtype=float)
    for mask in [frame[lr_col] < 1, frame[lr_col] > 1]:
        values = frame.loc[mask, lr_col].to_numpy(dtype=float)
        if len(values):
            predictions.loc[mask] = geometric_mean(values)
    frame[out_col] = predictions.where(predictions > 0)
    frame[f"ln_{out_col}"] = np.log(frame[out_col])


def add_sign_only_baseline_loo(frame: pd.DataFrame, lr_col: str, out_col: str) -> None:
    predictions = pd.Series(np.nan, index=frame.index, dtype=float)
    for mask in [frame[lr_col] < 1, frame[lr_col] > 1]:
        values = frame.loc[mask, lr_col].to_numpy(dtype=float)
        predictions.loc[mask] = leave_one_out_geometric_mean(values)
    frame[out_col] = predictions.where(predictions > 0)
    frame[f"ln_{out_col}"] = np.log(frame[out_col])


def add_source_label_baseline_constant(
    frame: pd.DataFrame,
    label_col: str,
    lr_col: str,
    out_col: str,
) -> None:
    predictions = pd.Series(np.nan, index=frame.index, dtype=float)
    for label in SOURCE_LABEL_ORDER:
        mask = frame[label_col] == label
        values = frame.loc[mask, lr_col].to_numpy(dtype=float)
        if len(values):
            predictions.loc[mask] = geometric_mean(values)
    frame[out_col] = predictions.where(predictions > 0)
    frame[f"ln_{out_col}"] = np.log(frame[out_col])


def add_source_label_baseline_loo(
    frame: pd.DataFrame,
    label_col: str,
    lr_col: str,
    out_col: str,
) -> None:
    predictions = pd.Series(np.nan, index=frame.index, dtype=float)
    for label in SOURCE_LABEL_ORDER:
        mask = frame[label_col] == label
        values = frame.loc[mask, lr_col].to_numpy(dtype=float)
        predictions.loc[mask] = leave_one_out_geometric_mean(values)
    frame[out_col] = predictions.where(predictions > 0)
    frame[f"ln_{out_col}"] = np.log(frame[out_col])


def add_sign_feature_baseline_constant(
    frame: pd.DataFrame,
    lr_col: str,
    feature_col: str,
    fallback_col: str,
    out_col: str,
    min_group_size: int = 10,
) -> None:
    feature = (
        frame[feature_col]
        .fillna("Missing")
        .astype(str)
        .str.strip()
        .replace({"": "Missing"})
    )
    predictions = pd.Series(np.nan, index=frame.index, dtype=float)

    sign_labels = pd.Series(
        np.select(
            [frame[lr_col] < 1, frame[lr_col] > 1],
            ["negative", "positive"],
            default="neutral",
        ),
        index=frame.index,
    )

    group_df = pd.DataFrame({"sign": sign_labels, "feature": feature}, index=frame.index)
    for (sign, feat), idx in group_df.groupby(["sign", "feature"], observed=True).groups.items():
        idx = pd.Index(idx)
        if sign == "neutral":
            continue
        values = frame.loc[idx, lr_col].to_numpy(dtype=float)
        if len(values) >= min_group_size:
            predictions.loc[idx] = geometric_mean(values)

    predictions = predictions.fillna(frame[fallback_col])
    frame[out_col] = predictions.where(predictions > 0)
    frame[f"ln_{out_col}"] = np.log(frame[out_col])


def add_sign_feature_baseline_loo(
    frame: pd.DataFrame,
    lr_col: str,
    feature_col: str,
    fallback_col: str,
    out_col: str,
    min_group_size: int = 10,
) -> None:
    feature = (
        frame[feature_col]
        .fillna("Missing")
        .astype(str)
        .str.strip()
        .replace({"": "Missing"})
    )
    predictions = pd.Series(np.nan, index=frame.index, dtype=float)

    sign_labels = pd.Series(
        np.select(
            [frame[lr_col] < 1, frame[lr_col] > 1],
            ["negative", "positive"],
            default="neutral",
        ),
        index=frame.index,
    )

    group_df = pd.DataFrame({"sign": sign_labels, "feature": feature}, index=frame.index)
    for (sign, feat), idx in group_df.groupby(["sign", "feature"], observed=True).groups.items():
        idx = pd.Index(idx)
        if sign == "neutral":
            continue
        values = frame.loc[idx, lr_col].to_numpy(dtype=float)
        if len(values) >= min_group_size and len(values) > 1:
            predictions.loc[idx] = leave_one_out_geometric_mean(values)

    predictions = predictions.fillna(frame[fallback_col])
    frame[out_col] = predictions.where(predictions > 0)
    frame[f"ln_{out_col}"] = np.log(frame[out_col])


def weight_matrix(k: int, scheme: str = "quadratic") -> np.ndarray:
    distance = np.abs(np.subtract.outer(np.arange(k), np.arange(k))).astype(float)
    if k <= 1:
        return np.ones((k, k), dtype=float)
    if scheme == "linear":
        return 1.0 - distance / (k - 1)
    return 1.0 - (distance / (k - 1)) ** 2


def weighted_kappa_with_ci(
    y_true: pd.Categorical,
    y_pred: pd.Categorical,
    scheme: str = "quadratic",
    alpha: float = 0.05,
) -> tuple[float, float, float]:
    true_codes = pd.Categorical(y_true, categories=QUAL_LABELS, ordered=True).codes
    pred_codes = pd.Categorical(y_pred, categories=QUAL_LABELS, ordered=True).codes
    mask = (true_codes >= 0) & (pred_codes >= 0)
    if mask.sum() == 0:
        return np.nan, np.nan, np.nan

    cm = confusion_matrix(true_codes[mask], pred_codes[mask], labels=np.arange(len(QUAL_LABELS)))
    try:
        result = cohens_kappa(cm, wt=scheme)
    except TypeError:
        result = cohens_kappa(cm, weights=weight_matrix(cm.shape[0], scheme))

    if hasattr(result, "kappa"):
        kappa = float(result.kappa)
        if hasattr(result, "se_kappa") and np.isfinite(result.se_kappa):
            se = float(result.se_kappa)
        elif hasattr(result, "var_kappa") and np.isfinite(result.var_kappa):
            se = float(np.sqrt(result.var_kappa))
        else:
            se = np.nan
    elif isinstance(result, (tuple, list)) and len(result) >= 2:
        kappa = float(result[0])
        se = float(np.sqrt(result[1])) if np.isfinite(result[1]) else np.nan
    else:
        kappa = float(result)
        se = np.nan

    if np.isfinite(se):
        z = stats.norm.ppf(1 - alpha / 2)
        low = max(-1.0, kappa - z * se)
        high = min(1.0, kappa + z * se)
        return kappa, low, high

    return kappa, np.nan, np.nan


def posterior_probability(pretest_probability: float, lr_values: pd.Series | np.ndarray) -> np.ndarray:
    lr_values = np.asarray(lr_values, dtype=float)
    pretest_odds = pretest_probability / (1 - pretest_probability)
    posttest_odds = pretest_odds * lr_values
    return posttest_odds / (1 + posttest_odds)


## 3. Overall calibration and association


In [ ]:
overall_rows = []
overall_pairs = []
for spec in MODEL_SPECS:
    overall_rows.append(summarize_method(df, spec, stratum="Overall"))
    overall_pairs.append(paired_xy(df, "ln_lr_reported", spec["ln_col"]))

overall_metrics = sort_methods(pd.DataFrame(overall_rows))
overall_metrics_path = FOLLOWUP_DIR / "Supp table - overall calibration and association.csv"
overall_metrics.to_csv(overall_metrics_path, index=False)

overall_limits = shared_axis_limits(overall_pairs)
fig, axes = plt.subplots(1, len(MODEL_SPECS), figsize=(15, 5), sharex=True, sharey=True)
for ax, spec in zip(axes, MODEL_SPECS):
    x, y = paired_xy(df, "ln_lr_reported", spec["ln_col"])
    plot_calibration_panel(ax, x, y, title=spec["label"], color=spec["color"], limits=overall_limits)
for ax in axes[1:]:
    ax.set_ylabel("")
fig.legend(
    handles=calibration_legend_handles(MODEL_SPECS[0]["color"]),
    loc="lower center",
    ncol=6,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
)
fig.suptitle("Overall calibration on the natural-log LR scale", y=1.02)
fig.tight_layout(rect=(0, 0.06, 1, 1))
overall_fig_path = FOLLOWUP_DIR / "Supp fig - overall calibration 3 panel.pdf"
fig.savefig(overall_fig_path)
plt.close(fig)

print(f"Saved figure -> {overall_fig_path}")
print(f"Saved table  -> {overall_metrics_path}")
display(
    overall_metrics[
        [
            "method",
            "n",
            "intercept",
            "intercept_ci_low",
            "intercept_ci_high",
            "slope",
            "slope_ci_low",
            "slope_ci_high",
            "r2",
            "pearson_r",
            "spearman_rho",
            "mean_log_error",
            "median_abs_log_error",
            "rmse_log",
            "typical_error_factor",
            "ccc_log",
        ]
    ].round(3)
)


## 4. Within-stratum calibration and association


In [ ]:
PRIMARY_STRATA = [
    ("LR < 1", df["lr_reported"] < 1),
    ("LR > 1", df["lr_reported"] > 1),
]

within_rows = []
within_pairs = []
for stratum_name, stratum_mask in PRIMARY_STRATA:
    for spec in MODEL_SPECS:
        within_rows.append(summarize_method(df, spec, stratum=stratum_name, mask=stratum_mask))
        within_pairs.append(paired_xy(df, "ln_lr_reported", spec["ln_col"], mask=stratum_mask))

within_metrics = sort_methods(pd.DataFrame(within_rows))
within_metrics_path = FOLLOWUP_DIR / "Supp table - within-stratum calibration and association.csv"
within_metrics.to_csv(within_metrics_path, index=False)

within_limits = shared_axis_limits(within_pairs)
fig, axes = plt.subplots(2, len(MODEL_SPECS), figsize=(15, 9), sharex=True, sharey=True)
for row_idx, (stratum_name, stratum_mask) in enumerate(PRIMARY_STRATA):
    for col_idx, spec in enumerate(MODEL_SPECS):
        ax = axes[row_idx, col_idx]
        x, y = paired_xy(df, "ln_lr_reported", spec["ln_col"], mask=stratum_mask)
        plot_calibration_panel(
            ax,
            x,
            y,
            title=spec["label"] if row_idx == 0 else "",
            color=spec["color"],
            limits=within_limits,
        )
        if row_idx == 0:
            ax.set_title(spec["label"])
        if col_idx == 0:
            ax.set_ylabel(f"{stratum_name}\nLLM-generated ln(LR)")
        else:
            ax.set_ylabel("")
fig.legend(
    handles=calibration_legend_handles(MODEL_SPECS[0]["color"]),
    loc="lower center",
    ncol=6,
    frameon=False,
    bbox_to_anchor=(0.5, -0.01),
)
fig.suptitle("Within-stratum calibration on the natural-log LR scale", y=1.01)
fig.tight_layout(rect=(0, 0.05, 1, 0.99))
within_fig_path = FOLLOWUP_DIR / "Supp fig - within-stratum calibration 2x3.pdf"
fig.savefig(within_fig_path)
plt.close(fig)

print(f"Saved figure -> {within_fig_path}")
print(f"Saved table  -> {within_metrics_path}")
display(
    within_metrics[
        [
            "stratum",
            "method",
            "n",
            "intercept",
            "slope",
            "r2",
            "pearson_r",
            "spearman_rho",
            "mean_log_error",
            "median_abs_log_error",
            "rmse_log",
            "typical_error_factor",
            "ccc_log",
        ]
    ].round(3)
)

if DIRECTION_COLUMN is not None:
    direction_series = df[DIRECTION_COLUMN].fillna("Missing").astype(str).str.strip().replace({"": "Missing"})
    direction_rows = []
    for direction_value in sorted(direction_series.unique()):
        direction_mask = direction_series == direction_value
        for spec in MODEL_SPECS:
            direction_rows.append(
                summarize_method(
                    df,
                    spec,
                    stratum=f"{DIRECTION_COLUMN} = {direction_value}",
                    mask=direction_mask,
                )
            )
    direction_metrics = sort_methods(pd.DataFrame(direction_rows))
    direction_metrics_path = FOLLOWUP_DIR / "Optional table - explicit direction calibration and association.csv"
    direction_metrics.to_csv(direction_metrics_path, index=False)
    print(f"Saved optional explicit-direction table -> {direction_metrics_path}")
else:
    print("No explicit present/absent or positive/negative direction column was detected; LR-based strata are evaluated here, and source-derived true LR+ / LR− strata are evaluated in Section 9.")



## Reliability-zone analysis

This requested analysis compares model performance in a central LR range (`0.2 <= reported LR <= 5.0`) versus more extreme reported LRs (`reported LR < 0.2` or `reported LR > 5.0`). The metrics reuse the same log-scale calibration, error, and Bland-Altman summaries used elsewhere in this supplement.

In [ ]:
RELIABILITY_ZONE_SPECS = [
    {
        "zone": "Central reliability zone",
        "reported_lr_range": "0.2 <= reported LR <= 5.0",
        "mask": df["lr_reported"].between(0.2, 5.0, inclusive="both"),
        "interpretation": "Central range where calibration curves suggest greatest reliability.",
    },
    {
        "zone": "Extreme LR zone",
        "reported_lr_range": "reported LR < 0.2 or reported LR > 5.0",
        "mask": (df["lr_reported"] < 0.2) | (df["lr_reported"] > 5.0),
        "interpretation": "Extreme range where model-generated outputs may compress toward neutrality and should be interpreted cautiously.",
    },
]

reliability_zone_rows = []
for zone_spec in RELIABILITY_ZONE_SPECS:
    for spec in MODEL_SPECS:
        row = summarize_method(
            df,
            spec,
            stratum=zone_spec["zone"],
            mask=zone_spec["mask"],
        )
        row.update(
            {
                "zone": zone_spec["zone"],
                "reported_lr_range": zone_spec["reported_lr_range"],
                "interpretation": zone_spec["interpretation"],
            }
        )
        reliability_zone_rows.append(row)

reliability_zone_metrics = sort_methods(pd.DataFrame(reliability_zone_rows))
reliability_zone_columns = [
    "zone",
    "reported_lr_range",
    "method",
    "kind",
    "prediction_variant",
    "stratum",
    "n",
    "intercept",
    "intercept_ci_low",
    "intercept_ci_high",
    "slope",
    "slope_ci_low",
    "slope_ci_high",
    "r2",
    "pearson_r",
    "pearson_p",
    "spearman_rho",
    "spearman_p",
    "mean_log_error",
    "median_log_error",
    "mean_abs_log_error",
    "median_abs_log_error",
    "rmse_log",
    "typical_error_factor",
    "rmse_error_factor",
    "mean_bias_xfold",
    "loa95_lower_xfold",
    "loa95_upper_xfold",
    "ccc_log",
    "interpretation",
]
reliability_zone_metrics = reliability_zone_metrics[reliability_zone_columns]

reliability_zone_path = FOLLOWUP_DIR / "Supp table - reliability zones central vs extreme.csv"
reliability_zone_metrics.to_csv(reliability_zone_path, index=False)

print(f"Saved table -> {reliability_zone_path}")
display(
    reliability_zone_metrics[
        [
            "zone",
            "reported_lr_range",
            "method",
            "n",
            "slope",
            "slope_ci_low",
            "slope_ci_high",
            "r2",
            "median_abs_log_error",
            "rmse_log",
            "typical_error_factor",
            "mean_bias_xfold",
            "loa95_lower_xfold",
            "loa95_upper_xfold",
        ]
    ].round(3)
)

## Laboratory discrepancy table

This descriptive requested table ranks the largest GPT-5 discrepancies among lab-like test-result findings. Flags are derived only from the curated finding text; no primary-study abstraction or causal error taxonomy is attempted.


In [ ]:
import re

LAB_KEYWORD_PATTERN = re.compile(
    r"\b(?:"
    r"bnp|nt[- ]?probnp|troponin|d[- ]?dimer|rpr|rapid plasma reagin|"
    r"esr|erythrocyte sedimentation|c reactive protein|crp|"
    r"hemoglobin|glucose|wbc|white blood|platelet|creatinine|lactate|"
    r"bilirubin|aminotransferase|alt|ast|lipase|amylase|urinalysis|"
    r"proteinuria|ketone|hba1c|serum|plasma|blood|urine|assay|culture|"
    r"antibody|antigen|titer|mg/dl|pg/ml|ng/ml|mm/h|mm/hr|u/l|iu/l|g/dl"
    r")\b|%",
    flags=re.IGNORECASE,
)
THRESHOLD_PATTERN = re.compile(
    r"(?:[<>]=?|>=|<=|≥|≤|\bat least\b|\babove\b|\bbelow\b)\s*\d|\d+(?:\.\d+)?",
    flags=re.IGNORECASE,
)
UNIT_PATTERN = re.compile(
    r"(?:mg\s*/\s*dL|pg\s*/\s*mL|ng\s*/\s*mL|mm\s*/\s*h|mm\s*/\s*hr|"
    r"u\s*/\s*L|iu\s*/\s*L|g\s*/\s*dL|mmol\s*/\s*L|cells?\s*/\s*(?:uL|µL|μL)|%)",
    flags=re.IGNORECASE,
)

CONDITION_LABEL_WORKBOOK = WORKING_DIR / "nnt_lrs_with_estimated.xlsx"


def load_condition_label_map(path: Path) -> dict[str, str]:
    if not path.exists():
        return {}
    label_map: dict[str, str] = {}
    workbook = pd.ExcelFile(path)
    for sheet_name in workbook.sheet_names:
        if sheet_name == SHEET_NAME:
            continue
        try:
            first_cell = pd.read_excel(
                path,
                sheet_name=sheet_name,
                header=None,
                nrows=1,
            ).iloc[0, 0]
        except Exception:
            continue
        if isinstance(first_cell, str) and first_cell.strip():
            label_map[sheet_name] = first_cell.strip()
    return label_map


def full_condition_label(value, label_map: dict[str, str]) -> str:
    if pd.isna(value):
        return ""
    raw = str(value).strip()
    return label_map.get(raw, raw)


condition_label_map = load_condition_label_map(CONDITION_LABEL_WORKBOOK)
if CONDITION_COLUMN is not None:
    laboratory_condition = df[CONDITION_COLUMN].map(lambda value: full_condition_label(value, condition_label_map))
else:
    laboratory_condition = pd.Series([""] * len(df), index=df.index)

finding_text = df[FINDING_COLUMN].fillna("").astype(str)
feature_type_text = df["Feature Type"].fillna("").astype(str)
lab_like_mask = feature_type_text.str.contains("test", case=False, na=False) & finding_text.str.contains(
    LAB_KEYWORD_PATTERN,
    na=False,
)
valid_gpt5_lr_mask = df[["lr_reported", "lr_gpt-5"]].notna().all(axis=1) & (df["lr_reported"] > 0) & (df["lr_gpt-5"] > 0)

laboratory_discrepancy_candidates = df.loc[lab_like_mask & valid_gpt5_lr_mask].copy()
laboratory_discrepancy_candidates["condition"] = laboratory_condition.loc[laboratory_discrepancy_candidates.index]
laboratory_discrepancy_candidates["finding"] = finding_text.loc[laboratory_discrepancy_candidates.index]
laboratory_discrepancy_candidates["feature_type"] = feature_type_text.loc[laboratory_discrepancy_candidates.index]
laboratory_discrepancy_candidates["lr_gpt5"] = laboratory_discrepancy_candidates["lr_gpt-5"]
laboratory_discrepancy_candidates["abs_log_error"] = (
    np.log(laboratory_discrepancy_candidates["lr_gpt5"]) - np.log(laboratory_discrepancy_candidates["lr_reported"])
).abs()
laboratory_discrepancy_candidates["threshold_present"] = laboratory_discrepancy_candidates["finding"].str.contains(
    THRESHOLD_PATTERN,
    na=False,
)
laboratory_discrepancy_candidates["unit_present"] = laboratory_discrepancy_candidates["finding"].str.contains(
    UNIT_PATTERN,
    na=False,
)
laboratory_discrepancy_candidates["extreme_reported_lr"] = (
    (laboratory_discrepancy_candidates["lr_reported"] < 0.2)
    | (laboratory_discrepancy_candidates["lr_reported"] > 5.0)
)
laboratory_discrepancy_candidates["notes"] = "Flags from curated finding text only; no causal category assigned."

laboratory_discrepancy_top10 = (
    laboratory_discrepancy_candidates.sort_values("abs_log_error", ascending=False)
    .head(10)
    .copy()
)
laboratory_discrepancy_top10.insert(0, "rank", range(1, len(laboratory_discrepancy_top10) + 1))

laboratory_discrepancy_columns = [
    "rank",
    "condition",
    "finding",
    "feature_type",
    "lr_reported",
    "lr_gpt5",
    "abs_log_error",
    "threshold_present",
    "unit_present",
    "extreme_reported_lr",
    "notes",
]
laboratory_discrepancy_top10 = laboratory_discrepancy_top10[laboratory_discrepancy_columns]

if laboratory_discrepancy_top10.shape[0] != 10:
    raise ValueError(
        "Expected 10 rows in the GPT-5 laboratory discrepancy table, "
        f"found {laboratory_discrepancy_top10.shape[0]}."
    )

expected_lab_flag_counts = {
    "threshold_present": 5,
    "unit_present": 3,
    "extreme_reported_lr": 6,
}
observed_lab_flag_counts = {
    column: int(laboratory_discrepancy_top10[column].sum())
    for column in expected_lab_flag_counts
}
if observed_lab_flag_counts != expected_lab_flag_counts:
    raise ValueError(
        "Unexpected GPT-5 laboratory discrepancy flag counts. "
        f"Expected {expected_lab_flag_counts}, observed {observed_lab_flag_counts}."
    )

laboratory_discrepancy_path = FOLLOWUP_DIR / "Supp table - GPT5 laboratory discrepancies.csv"
laboratory_discrepancy_top10.to_csv(laboratory_discrepancy_path, index=False)

print(f"Lab-like valid GPT-5 candidate rows: {laboratory_discrepancy_candidates.shape[0]}")
print(f"Saved table -> {laboratory_discrepancy_path}")
print(f"Top-10 flag counts -> {observed_lab_flag_counts}")
display(laboratory_discrepancy_top10)


## 5. Trivial heuristic baseline(s)


In [ ]:
print(
    "Primary heuristic comparisons use pure constant baselines within sign or sign x feature strata. "
    "Exact leave-one-out variants are also computed as sensitivity analyses in the baseline table, "
    "but are not used for the calibration/association comparison figures because the exact LOO construction "
    "induces mechanical row-level dependence within a stratum."
)

add_sign_only_baseline_constant(df, lr_col="lr_reported", out_col="lr_sign_only_baseline")
add_sign_only_baseline_loo(df, lr_col="lr_reported", out_col="lr_sign_only_baseline_loo")

BASELINE_SPECS = [
    {
        "label": "Sign-only heuristic",
        "kind": "baseline",
        "lr_col": "lr_sign_only_baseline",
        "ln_col": "ln_lr_sign_only_baseline",
        "color": "#6d6d6d",
        "variant": "constant",
    }
]
BASELINE_SENSITIVITY_SPECS = [
    {
        "label": "Sign-only heuristic",
        "kind": "baseline",
        "lr_col": "lr_sign_only_baseline_loo",
        "ln_col": "ln_lr_sign_only_baseline_loo",
        "color": "#6d6d6d",
        "variant": "loo_sensitivity",
    }
]

add_sign_feature_baseline_constant(
    df,
    lr_col="lr_reported",
    feature_col="Feature Type",
    fallback_col="lr_sign_only_baseline",
    out_col="lr_sign_feature_baseline",
    min_group_size=10,
)
add_sign_feature_baseline_loo(
    df,
    lr_col="lr_reported",
    feature_col="Feature Type",
    fallback_col="lr_sign_only_baseline_loo",
    out_col="lr_sign_feature_baseline_loo",
    min_group_size=10,
)
if df["lr_sign_feature_baseline"].notna().sum() > 0:
    BASELINE_SPECS.append(
        {
            "label": "Sign + feature-type heuristic",
            "kind": "baseline",
            "lr_col": "lr_sign_feature_baseline",
            "ln_col": "ln_lr_sign_feature_baseline",
            "color": "#8a8a8a",
            "variant": "constant",
        }
    )
if df["lr_sign_feature_baseline_loo"].notna().sum() > 0:
    BASELINE_SENSITIVITY_SPECS.append(
        {
            "label": "Sign + feature-type heuristic",
            "kind": "baseline",
            "lr_col": "lr_sign_feature_baseline_loo",
            "ln_col": "ln_lr_sign_feature_baseline_loo",
            "color": "#8a8a8a",
            "variant": "loo_sensitivity",
        }
    )

baseline_rows = []
for spec in BASELINE_SPECS + BASELINE_SENSITIVITY_SPECS:
    baseline_rows.append(summarize_method(df, spec, stratum="Overall"))
    for stratum_name, stratum_mask in PRIMARY_STRATA:
        baseline_rows.append(summarize_method(df, spec, stratum=stratum_name, mask=stratum_mask))

baseline_metrics = sort_methods(pd.DataFrame(baseline_rows))
baseline_metrics_path = FOLLOWUP_DIR / "Supp table - baseline performance.csv"
baseline_metrics.to_csv(baseline_metrics_path, index=False)

sign_only_spec = next(spec for spec in BASELINE_SPECS if spec["label"] == "Sign-only heuristic")
sign_only_panels = [
    ("Overall", pd.Series(True, index=df.index)),
    ("LR < 1", df["lr_reported"] < 1),
    ("LR > 1", df["lr_reported"] > 1),
]
sign_pairs = [paired_xy(df, "ln_lr_reported", sign_only_spec["ln_col"], mask=mask) for _, mask in sign_only_panels]
sign_limits = shared_axis_limits(sign_pairs)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
for ax, (panel_title, mask) in zip(axes, sign_only_panels):
    x, y = paired_xy(df, "ln_lr_reported", sign_only_spec["ln_col"], mask=mask)
    plot_calibration_panel(
        ax,
        x,
        y,
        title=panel_title,
        color=sign_only_spec["color"],
        limits=sign_limits,
    )
for ax in axes[1:]:
    ax.set_ylabel("")
fig.legend(
    handles=calibration_legend_handles(sign_only_spec["color"]),
    loc="lower center",
    ncol=6,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
)
fig.suptitle("Sign-only heuristic calibration", y=1.02)
fig.tight_layout(rect=(0, 0.06, 1, 1))
sign_only_fig_path = FOLLOWUP_DIR / "Supp fig - sign-only baseline calibration.pdf"
fig.savefig(sign_only_fig_path)
plt.close(fig)

if any(spec["label"] == "Sign + feature-type heuristic" for spec in BASELINE_SPECS):
    sign_feature_spec = next(spec for spec in BASELINE_SPECS if spec["label"] == "Sign + feature-type heuristic")
    feature_pairs = [paired_xy(df, "ln_lr_reported", sign_feature_spec["ln_col"], mask=mask) for _, mask in sign_only_panels]
    feature_limits = shared_axis_limits(feature_pairs)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
    for ax, (panel_title, mask) in zip(axes, sign_only_panels):
        x, y = paired_xy(df, "ln_lr_reported", sign_feature_spec["ln_col"], mask=mask)
        plot_calibration_panel(
            ax,
            x,
            y,
            title=panel_title,
            color=sign_feature_spec["color"],
            limits=feature_limits,
        )
    for ax in axes[1:]:
        ax.set_ylabel("")
    fig.legend(
        handles=calibration_legend_handles(sign_feature_spec["color"]),
        loc="lower center",
        ncol=6,
        frameon=False,
        bbox_to_anchor=(0.5, -0.02),
    )
    fig.suptitle("Sign plus feature-type heuristic calibration", y=1.02)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    feature_fig_path = FOLLOWUP_DIR / "Supp fig - sign-plus-feature baseline calibration.pdf"
    fig.savefig(feature_fig_path)
    plt.close(fig)
    print(f"Saved optional figure -> {feature_fig_path}")

print(f"Saved figure -> {sign_only_fig_path}")
print(f"Saved table  -> {baseline_metrics_path}")
display(
    baseline_metrics[
        [
            "stratum",
            "method",
            "prediction_variant",
            "n",
            "intercept",
            "slope",
            "r2",
            "pearson_r",
            "spearman_rho",
            "mean_log_error",
            "median_abs_log_error",
            "rmse_log",
            "typical_error_factor",
            "ccc_log",
        ]
    ].round(3)
)


## 6. Baseline-vs-model comparison tables


In [ ]:
comparison_columns = [
    "method",
    "kind",
    "n",
    "slope",
    "slope_ci_low",
    "slope_ci_high",
    "r2",
    "pearson_r",
    "spearman_rho",
    "median_abs_log_error",
    "rmse_log",
    "typical_error_factor",
    "ccc_log",
]

baseline_primary_metrics = baseline_metrics[baseline_metrics["prediction_variant"] == "constant"].copy()

model_vs_heuristic_overall = sort_methods(
    pd.concat(
        [
            overall_metrics.assign(stratum="Overall"),
            baseline_primary_metrics[baseline_primary_metrics["stratum"] == "Overall"],
        ],
        ignore_index=True,
    )
)
overall_comparison_path = FOLLOWUP_DIR / "Supp table - model vs heuristic comparison overall.csv"
model_vs_heuristic_overall[comparison_columns].to_csv(overall_comparison_path, index=False)

model_vs_heuristic_by_stratum = sort_methods(
    pd.concat(
        [
            within_metrics,
            baseline_primary_metrics[baseline_primary_metrics["stratum"].isin(["LR < 1", "LR > 1"])],
        ],
        ignore_index=True,
    )
)
by_stratum_comparison_path = FOLLOWUP_DIR / "Supp table - model vs heuristic comparison by stratum.csv"
model_vs_heuristic_by_stratum[["stratum"] + comparison_columns].to_csv(by_stratum_comparison_path, index=False)

print(f"Saved table -> {overall_comparison_path}")
print(f"Saved table -> {by_stratum_comparison_path}")

display(model_vs_heuristic_overall[comparison_columns].round(3))
display(model_vs_heuristic_by_stratum[["stratum"] + comparison_columns].round(3))


## 7. Qualitative-tier comparison versus baseline


In [ ]:
qualitative_reference = pd.Categorical(bin_qualitative_lr(df["lr_reported"]), categories=QUAL_LABELS, ordered=True)

agreement_rows = []
for spec in MODEL_SPECS + BASELINE_SPECS:
    qualitative_pred = pd.Categorical(bin_qualitative_lr(df[spec["lr_col"]]), categories=QUAL_LABELS, ordered=True)
    true_codes = qualitative_reference.codes
    pred_codes = qualitative_pred.codes
    mask = (true_codes >= 0) & (pred_codes >= 0)
    n_pairs = int(mask.sum())

    if n_pairs == 0:
        agreement_rows.append(
            {
                "method": spec["label"],
                "kind": spec["kind"],
                "n": 0,
                "weighted_kappa": np.nan,
                "weighted_kappa_ci_low": np.nan,
                "weighted_kappa_ci_high": np.nan,
                "exact_match_pct": np.nan,
                "within_1_tier_pct": np.nan,
            }
        )
        continue

    kappa, kappa_low, kappa_high = weighted_kappa_with_ci(
        qualitative_reference[mask],
        qualitative_pred[mask],
        scheme="quadratic",
    )
    exact_match = float(np.mean(true_codes[mask] == pred_codes[mask]) * 100)
    within_one = float(np.mean(np.abs(true_codes[mask] - pred_codes[mask]) <= 1) * 100)

    agreement_rows.append(
        {
            "method": spec["label"],
            "kind": spec["kind"],
            "n": n_pairs,
            "weighted_kappa": kappa,
            "weighted_kappa_ci_low": kappa_low,
            "weighted_kappa_ci_high": kappa_high,
            "exact_match_pct": exact_match,
            "within_1_tier_pct": within_one,
        }
    )

qualitative_agreement = sort_methods(pd.DataFrame(agreement_rows))
qualitative_path = FOLLOWUP_DIR / "Supp table - qualitative tier agreement vs baseline.csv"
qualitative_agreement.to_csv(qualitative_path, index=False)

print(f"Saved table -> {qualitative_path}")
display(qualitative_agreement.round(3))


## 8. Optional clinical-impact sensitivity analysis


In [ ]:
PRETEST_PROBABILITIES = [0.01, 0.05, 0.10, 0.25, 0.50]
CLINICAL_THRESHOLDS = [0.02, 0.10, 0.20]

posterior_rows = []
for pretest_probability in PRETEST_PROBABILITIES:
    reported_mask = df["lr_reported"].notna()
    reported_posterior = posterior_probability(pretest_probability, df.loc[reported_mask, "lr_reported"])

    for spec in MODEL_SPECS:
        mask = reported_mask & df[spec["lr_col"]].notna()
        model_posterior = posterior_probability(pretest_probability, df.loc[mask, spec["lr_col"]])
        reference_posterior = posterior_probability(pretest_probability, df.loc[mask, "lr_reported"])
        abs_difference = np.abs(model_posterior - reference_posterior)

        row = {
            "method": spec["label"],
            "pretest_probability": pretest_probability,
            "n": int(len(abs_difference)),
            "median_abs_posterior_diff": float(np.median(abs_difference)),
            "p90_abs_posterior_diff": float(np.quantile(abs_difference, 0.90)),
            "p95_abs_posterior_diff": float(np.quantile(abs_difference, 0.95)),
        }
        for threshold in CLINICAL_THRESHOLDS:
            row[f"flip_rate_at_{threshold:.2f}"] = float(
                np.mean((reference_posterior >= threshold) != (model_posterior >= threshold))
            )
        posterior_rows.append(row)

posterior_impact = sort_methods(pd.DataFrame(posterior_rows))
posterior_path = FOLLOWUP_DIR / "Supp table - posterior impact sensitivity.csv"
posterior_impact.to_csv(posterior_path, index=False)

fig, axes = plt.subplots(1, len(MODEL_SPECS), figsize=(15, 4.5), sharey=True)
for ax, spec in zip(axes, MODEL_SPECS):
    sub = posterior_impact[posterior_impact["method"] == spec["label"]].sort_values("pretest_probability")
    ax.plot(
        sub["pretest_probability"],
        sub["median_abs_posterior_diff"],
        marker="o",
        linewidth=1.8,
        color=spec["color"],
        label="Median",
    )
    ax.plot(
        sub["pretest_probability"],
        sub["p90_abs_posterior_diff"],
        marker="s",
        linewidth=1.4,
        linestyle="--",
        color=spec["color"],
        alpha=0.9,
        label="90th percentile",
    )
    ax.plot(
        sub["pretest_probability"],
        sub["p95_abs_posterior_diff"],
        marker="^",
        linewidth=1.2,
        linestyle=":",
        color="#222222",
        label="95th percentile",
    )
    ax.set_title(spec["label"])
    ax.set_xlabel("Pretest probability")
    ax.set_ylim(bottom=0)
axes[0].set_ylabel("Absolute posterior probability difference")
fig.legend(loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Posterior impact sensitivity by pretest probability", y=1.03)
fig.tight_layout(rect=(0, 0.06, 1, 1))
posterior_fig_path = FOLLOWUP_DIR / "Supp fig - posterior impact by pretest probability.pdf"
fig.savefig(posterior_fig_path)
plt.close(fig)

print(f"Saved figure -> {posterior_fig_path}")
print(f"Saved table  -> {posterior_path}")
display(posterior_impact.round(4))


## 9. True LR+ vs LR− stratified analysis



In [ ]:
SOURCE_LABEL_STRATA = [
    ("LR+", df["source_lr_label"] == "LR+"),
    ("LR−", df["source_lr_label"] == "LR−"),
]
SOURCE_LABEL_DISPLAY = {
    "LR+": "Finding Present",
    "LR−": "Finding Absent",
}
MODEL_TITLE_DISPLAY = {
    "GPT‑4o": "GPT-4o",
    "o3": "o3",
    "GPT‑5": "GPT-5",
}
REPORTED_STRATUM_ORDER = ["LR < 1", "LR = 1", "LR > 1"]

source_label_qc = pd.crosstab(df["source_lr_label"], df["reported_stratum"])
source_label_qc = source_label_qc.reindex(
    index=SOURCE_LABEL_ORDER,
    columns=REPORTED_STRATUM_ORDER,
    fill_value=0,
)
source_label_qc_export = source_label_qc.rename_axis(index="source_lr_label", columns="reported_stratum").reset_index()
source_label_qc_export["n_total"] = source_label_qc.sum(axis=1).to_numpy()
source_label_qc_export["pct_lr_lt_1"] = source_label_qc["LR < 1"].to_numpy() / source_label_qc_export["n_total"].to_numpy()
source_label_qc_export["pct_lr_eq_1"] = source_label_qc["LR = 1"].to_numpy() / source_label_qc_export["n_total"].to_numpy()
source_label_qc_export["pct_lr_gt_1"] = source_label_qc["LR > 1"].to_numpy() / source_label_qc_export["n_total"].to_numpy()
source_label_qc_export = source_label_qc_export.rename(
    columns={
        "LR < 1": "n_lr_lt_1",
        "LR = 1": "n_lr_eq_1",
        "LR > 1": "n_lr_gt_1",
    }
)
source_label_qc_path = FOLLOWUP_DIR / "Supp table - true LRplus vs LRminus cross-tab.csv"
source_label_qc_export.to_csv(source_label_qc_path, index=False)

true_label_rows = []
true_label_pairs = []
for stratum_name, stratum_mask in SOURCE_LABEL_STRATA:
    for spec in MODEL_SPECS:
        true_label_rows.append(
            summarize_method(df, spec, stratum=stratum_name, mask=stratum_mask)
        )
        true_label_pairs.append(
            paired_xy(df, "ln_lr_reported", spec["ln_col"], mask=stratum_mask)
        )

true_label_metrics = sort_methods(pd.DataFrame(true_label_rows))
true_label_metrics_path = FOLLOWUP_DIR / "Supp table - true LRplus vs LRminus calibration and association.csv"
true_label_metrics.to_csv(true_label_metrics_path, index=False)

true_label_limits = shared_axis_limits(true_label_pairs)
fig, axes = plt.subplots(2, len(MODEL_SPECS), figsize=(15, 9), sharex=True, sharey=True)
for row_idx, (stratum_name, stratum_mask) in enumerate(SOURCE_LABEL_STRATA):
    for col_idx, spec in enumerate(MODEL_SPECS):
        ax = axes[row_idx, col_idx]
        x, y = paired_xy(df, "ln_lr_reported", spec["ln_col"], mask=stratum_mask)
        plot_calibration_panel(
            ax,
            x,
            y,
            title=spec["label"] if row_idx == 0 else "",
            color=spec["color"],
            limits=true_label_limits,
        )
        if row_idx == 0:
            ax.set_title(MODEL_TITLE_DISPLAY.get(spec["label"], spec["label"]))
        if col_idx == 0:
            ax.set_ylabel(f"{SOURCE_LABEL_DISPLAY[stratum_name]}\nLLM-generated ln(LR)")
        else:
            ax.set_ylabel("")
fig.legend(
    handles=calibration_legend_handles(MODEL_SPECS[0]["color"]),
    loc="lower center",
    ncol=6,
    frameon=False,
    bbox_to_anchor=(0.5, -0.01),
)
fig.suptitle("Calibration within source-derived Finding Present and Finding Absent strata", y=1.01)
fig.tight_layout(rect=(0, 0.05, 1, 0.99))
true_label_fig_path = FOLLOWUP_DIR / "Supp fig - true LRplus vs LRminus calibration 2x3.pdf"
fig.savefig(true_label_fig_path)
plt.close(fig)

true_label_ba_x_limits, true_label_ba_y_limits = bland_altman_axis_limits(true_label_pairs)
fig, axes = plt.subplots(2, len(MODEL_SPECS), figsize=(12, 6.5), sharex=True, sharey=True)
for row_idx, (stratum_name, stratum_mask) in enumerate(SOURCE_LABEL_STRATA):
    for col_idx, spec in enumerate(MODEL_SPECS):
        ax = axes[row_idx, col_idx]
        x, y = paired_xy(df, "ln_lr_reported", spec["ln_col"], mask=stratum_mask)
        plot_bland_altman_panel(
            ax,
            x,
            y,
            color=spec["color"],
            x_limits=true_label_ba_x_limits,
            y_limits=true_label_ba_y_limits,
        )
        if row_idx == 0:
            ax.set_title(MODEL_TITLE_DISPLAY.get(spec["label"], spec["label"]))
        if col_idx == 0:
            ax.set_ylabel(f"{SOURCE_LABEL_DISPLAY[stratum_name]}\n\nLR ratio\n(Reported / Model)")
        else:
            ax.set_ylabel("")
        if row_idx == len(SOURCE_LABEL_STRATA) - 1:
            ax.set_xlabel("Geometric Mean LR")
for ax in axes.flat:
    ax.label_outer()
fig.suptitle("Bland-Altman agreement within source-derived Finding Present and Finding Absent strata", y=1.02)
fig.tight_layout(h_pad=1.2, w_pad=1.2)
true_label_ba_fig_path = FOLLOWUP_DIR / "Supp fig - true LRplus LRminus bland altman 2x3.pdf"
fig.savefig(true_label_ba_fig_path)
plt.close(fig)

print(
    "Literal LR+/LR− shortcut comparison uses a source-label-only heuristic based on the original scrape headings. "
    "The leave-one-out sensitivity variant is computed for completeness but excluded from the compact comparison table "
    "because it induces mechanical row-level dependence within each true-direction stratum."
)
add_source_label_baseline_constant(
    df,
    label_col="source_lr_label",
    lr_col="lr_reported",
    out_col="lr_source_label_only_baseline",
)
add_source_label_baseline_loo(
    df,
    label_col="source_lr_label",
    lr_col="lr_reported",
    out_col="lr_source_label_only_baseline_loo",
)

SOURCE_LABEL_BASELINE_SPECS = [
    {
        "label": "LR+/LR− label-only heuristic",
        "kind": "baseline",
        "lr_col": "lr_source_label_only_baseline",
        "ln_col": "ln_lr_source_label_only_baseline",
        "color": "#5b5b5b",
        "variant": "constant",
    },
    {
        "label": "LR+/LR− label-only heuristic",
        "kind": "baseline",
        "lr_col": "lr_source_label_only_baseline_loo",
        "ln_col": "ln_lr_source_label_only_baseline_loo",
        "color": "#5b5b5b",
        "variant": "loo_sensitivity",
    },
]

source_label_baseline_rows = []
for spec in SOURCE_LABEL_BASELINE_SPECS:
    source_label_baseline_rows.append(summarize_method(df, spec, stratum="Overall"))
    for stratum_name, stratum_mask in SOURCE_LABEL_STRATA:
        source_label_baseline_rows.append(
            summarize_method(df, spec, stratum=stratum_name, mask=stratum_mask)
        )

source_label_baseline_metrics = sort_methods(pd.DataFrame(source_label_baseline_rows))
source_label_baseline_primary = source_label_baseline_metrics[
    source_label_baseline_metrics["prediction_variant"] == "constant"
].copy()
source_label_comparison = sort_methods(
    pd.concat(
        [
            overall_metrics.assign(stratum="Overall"),
            true_label_metrics,
            source_label_baseline_primary,
        ],
        ignore_index=True,
    )
)
source_label_comparison_columns = [
    "stratum",
    "method",
    "kind",
    "prediction_variant",
    "n",
    "slope",
    "slope_ci_low",
    "slope_ci_high",
    "r2",
    "pearson_r",
    "spearman_rho",
    "median_abs_log_error",
    "rmse_log",
    "typical_error_factor",
    "mean_bias_xfold",
    "loa95_lower_xfold",
    "loa95_upper_xfold",
    "ccc_log",
]
source_label_comparison_path = FOLLOWUP_DIR / "Supp table - LRplus LRminus label-only heuristic.csv"
source_label_comparison[source_label_comparison_columns].to_csv(source_label_comparison_path, index=False)

print(f"Saved table  -> {source_label_qc_path}")
print(f"Saved figure -> {true_label_fig_path}")
print(f"Saved figure -> {true_label_ba_fig_path}")
print(f"Saved table  -> {true_label_metrics_path}")
print(f"Saved table  -> {source_label_comparison_path}")
display(source_label_qc_export.round(3))
display(
    true_label_metrics[
        [
            "stratum",
            "method",
            "n",
            "intercept",
            "slope",
            "r2",
            "pearson_r",
            "spearman_rho",
            "mean_bias_xfold",
            "loa95_lower_xfold",
            "loa95_upper_xfold",
            "mean_log_error",
            "median_abs_log_error",
            "rmse_log",
            "typical_error_factor",
            "ccc_log",
        ]
    ].round(3)
)
display(source_label_comparison[source_label_comparison_columns].round(3))


## 10. Manuscript-ready summary outputs



In [ ]:
model_overall = overall_metrics[overall_metrics["kind"] == "model"].copy()
model_within = within_metrics[within_metrics["kind"] == "model"].copy()
model_true_direction = true_label_metrics[true_label_metrics["kind"] == "model"].copy()
model_reliability = reliability_zone_metrics[reliability_zone_metrics["kind"] == "model"].copy()
baseline_primary_metrics = baseline_metrics[baseline_metrics["prediction_variant"] == "constant"].copy()
sign_only_overall = baseline_primary_metrics[
    (baseline_primary_metrics["method"] == "Sign-only heuristic") & (baseline_primary_metrics["stratum"] == "Overall")
].iloc[0]
sign_only_neg = baseline_primary_metrics[
    (baseline_primary_metrics["method"] == "Sign-only heuristic") & (baseline_primary_metrics["stratum"] == "LR < 1")
].iloc[0]
sign_only_pos = baseline_primary_metrics[
    (baseline_primary_metrics["method"] == "Sign-only heuristic") & (baseline_primary_metrics["stratum"] == "LR > 1")
].iloc[0]
source_label_counts_map = source_label_counts.set_index("source_lr_label")["n"]
source_label_qc_map = source_label_qc_export.set_index("source_lr_label")
source_label_primary_comparison = source_label_comparison[
    source_label_comparison["prediction_variant"].isin(["primary", "constant"])
].copy()
label_only_overall = source_label_primary_comparison[
    (source_label_primary_comparison["method"] == "LR+/LR− label-only heuristic")
    & (source_label_primary_comparison["stratum"] == "Overall")
    & (source_label_primary_comparison["prediction_variant"] == "constant")
].iloc[0]
label_only_lr_plus = source_label_primary_comparison[
    (source_label_primary_comparison["method"] == "LR+/LR− label-only heuristic")
    & (source_label_primary_comparison["stratum"] == "LR+")
    & (source_label_primary_comparison["prediction_variant"] == "constant")
].iloc[0]
label_only_lr_minus = source_label_primary_comparison[
    (source_label_primary_comparison["method"] == "LR+/LR− label-only heuristic")
    & (source_label_primary_comparison["stratum"] == "LR−")
    & (source_label_primary_comparison["prediction_variant"] == "constant")
].iloc[0]


def format_range(series: pd.Series, digits: int = 2) -> str:
    finite = pd.to_numeric(series, errors="coerce").dropna()
    if finite.empty:
        return "NA"
    return f"{finite.min():.{digits}f} to {finite.max():.{digits}f}"


def format_bland_altman_details(frame: pd.DataFrame, stratum: str) -> str:
    subset = frame[frame["stratum"] == stratum].copy()
    if subset.empty:
        return "NA"
    subset["method"] = pd.Categorical(subset["method"], categories=METHOD_ORDER, ordered=True)
    subset = subset.sort_values("method")
    details = []
    for _, row in subset.iterrows():
        details.append(
            f"{row['method']} mean bias {row['mean_bias_xfold']:.2f}x with 95% limits of agreement {row['loa95_lower_xfold']:.2f}x to {row['loa95_upper_xfold']:.2f}x"
        )
    return "; ".join(details)


def best_method_row(frame: pd.DataFrame, metric: str, ascending: bool) -> pd.Series:
    tmp = frame.dropna(subset=[metric]).sort_values(metric, ascending=ascending)
    return tmp.iloc[0] if not tmp.empty else pd.Series(dtype=float)


def zone_method_row(frame: pd.DataFrame, zone: str, method: str) -> pd.Series:
    subset = frame[(frame["zone"] == zone) & (frame["method"] == method)]
    return subset.iloc[0] if not subset.empty else pd.Series(dtype=float)


best_rmse_row = best_method_row(model_overall, "rmse_log", ascending=True)
best_r2_row = best_method_row(model_overall, "r2", ascending=False)
best_kappa_row = best_method_row(
    qualitative_agreement[qualitative_agreement["kind"] == "model"],
    "weighted_kappa",
    ascending=False,
)
sign_feature_row = qualitative_agreement[
    qualitative_agreement["method"] == "Sign + feature-type heuristic"
]

lr_plus_n = int(source_label_counts_map.get("LR+", 0))
lr_minus_n = int(source_label_counts_map.get("LR−", 0))
lr_plus_lt1 = int(source_label_qc_map.loc["LR+", "n_lr_lt_1"])
lr_plus_eq1 = int(source_label_qc_map.loc["LR+", "n_lr_eq_1"])
lr_plus_gt1 = int(source_label_qc_map.loc["LR+", "n_lr_gt_1"])
lr_minus_lt1 = int(source_label_qc_map.loc["LR−", "n_lr_lt_1"])
lr_minus_eq1 = int(source_label_qc_map.loc["LR−", "n_lr_eq_1"])
lr_minus_gt1 = int(source_label_qc_map.loc["LR−", "n_lr_gt_1"])
lr_plus_lt1_pct = float(source_label_qc_map.loc["LR+", "pct_lr_lt_1"])
lr_plus_eq1_pct = float(source_label_qc_map.loc["LR+", "pct_lr_eq_1"])
lr_minus_gt1_pct = float(source_label_qc_map.loc["LR−", "pct_lr_gt_1"])
lr_minus_eq1_pct = float(source_label_qc_map.loc["LR−", "pct_lr_eq_1"])
lr_plus_ba_details = format_bland_altman_details(model_true_direction, "LR+")
lr_minus_ba_details = format_bland_altman_details(model_true_direction, "LR−")
central_reliability = model_reliability[model_reliability["zone"] == "Central reliability zone"].copy()
extreme_reliability = model_reliability[model_reliability["zone"] == "Extreme LR zone"].copy()
gpt5_central_reliability = zone_method_row(model_reliability, "Central reliability zone", "GPT‑5")
gpt5_extreme_reliability = zone_method_row(model_reliability, "Extreme LR zone", "GPT‑5")
lab_threshold_count = int(laboratory_discrepancy_top10["threshold_present"].sum())
lab_unit_count = int(laboratory_discrepancy_top10["unit_present"].sum())
lab_extreme_count = int(laboratory_discrepancy_top10["extreme_reported_lr"].sum())

summary_lines = [
    "# Additional requested analyses summary",
    "",
    f"- Source workbook: `{WORKBOOK_PATH.name}` (`{SHEET_NAME}` sheet). Outputs were written to `{FOLLOWUP_DIR}`.",
    f"- Overall association on the natural-log LR scale was positive for all three models: Pearson r ranged from {format_range(model_overall['pearson_r'])}, Spearman rho ranged from {format_range(model_overall['spearman_rho'])}, and R^2 ranged from {format_range(model_overall['r2'])}.",
    f"- Positive association persisted within both primary LR-sign strata. For LR < 1, Pearson r ranged from {format_range(model_within.loc[model_within['stratum'] == 'LR < 1', 'pearson_r'])}; for LR > 1, Pearson r ranged from {format_range(model_within.loc[model_within['stratum'] == 'LR > 1', 'pearson_r'])}.",
    f"- Overall calibration slopes were below 1.0 for all models ({format_range(model_overall['slope'])}), consistent with shrinkage toward neutrality rather than one-for-one recovery of reported magnitudes.",
    f"- Mean log error was relatively small overall ({format_range(model_overall['mean_log_error'])}), but dispersion remained wide: median absolute log error corresponded to typical multiplicative error factors of {format_range(model_overall['typical_error_factor'])}x and RMSE factors of {format_range(model_overall['rmse_error_factor'])}x.",
    f"- Reliability-zone analysis supported greatest reliability in the central LR range (0.2 <= reported LR <= 5.0; n={int(gpt5_central_reliability['n'])} for GPT-5). Across models, median absolute log error was {format_range(central_reliability['median_abs_log_error'])} centrally versus {format_range(extreme_reliability['median_abs_log_error'])} in the extreme zone; RMSE_log was {format_range(central_reliability['rmse_log'])} versus {format_range(extreme_reliability['rmse_log'])}; and calibration slopes were {format_range(central_reliability['slope'])} versus {format_range(extreme_reliability['slope'])}. For GPT-5 specifically, median absolute log error was {gpt5_central_reliability['median_abs_log_error']:.2f} centrally versus {gpt5_extreme_reliability['median_abs_log_error']:.2f} in the extreme zone, with RMSE_log {gpt5_central_reliability['rmse_log']:.2f} versus {gpt5_extreme_reliability['rmse_log']:.2f} and slope {gpt5_central_reliability['slope']:.2f} versus {gpt5_extreme_reliability['slope']:.2f}.",
    f"- Descriptive GPT-5 laboratory discrepancy table: among the 10 largest GPT-5 discrepancies in lab-like test-result findings, {lab_threshold_count}/10 contained an explicit numeric threshold or comparator, {lab_unit_count}/10 contained a unit, and {lab_extreme_count}/10 had extreme reported LRs (<0.2 or >5.0). These flags are descriptive only and use curated finding text, not primary-study abstraction.",
    f"- Against the pure sign-only heuristic, the models had better overall RMSE_log ({format_range(model_overall['rmse_log'])} vs {sign_only_overall['rmse_log']:.2f}), R^2 ({format_range(model_overall['r2'])} vs {sign_only_overall['r2']:.2f}), and weighted qualitative agreement ({format_range(qualitative_agreement.loc[qualitative_agreement['kind'] == 'model', 'weighted_kappa'])} vs {qualitative_agreement.loc[qualitative_agreement['method'] == 'Sign-only heuristic', 'weighted_kappa'].iloc[0]:.2f}).",
    f"- The pure sign-only heuristic showed essentially no within-stratum association by construction: its within-stratum Pearson correlations were {sign_only_neg['pearson_r']:.2f} for LR < 1 and {sign_only_pos['pearson_r']:.2f} for LR > 1, whereas model within-stratum correlations remained positive.",
    f"- The strongest continuous overall fit in this dataset was {best_r2_row.get('method', 'NA')} by R^2 ({best_r2_row.get('r2', np.nan):.2f}), and the lowest overall RMSE_log was {best_rmse_row.get('method', 'NA')} ({best_rmse_row.get('rmse_log', np.nan):.2f}). The strongest qualitative-tier agreement was {best_kappa_row.get('method', 'NA')} with quadratic-weighted kappa {best_kappa_row.get('weighted_kappa', np.nan):.2f}.",
]

if not sign_feature_row.empty:
    sign_feature_row = sign_feature_row.iloc[0]
    summary_lines.append(
        f"- The optional sign + feature-type heuristic improved on sign-only qualitative agreement (weighted kappa {sign_feature_row['weighted_kappa']:.2f} vs {qualitative_agreement.loc[qualitative_agreement['method'] == 'Sign-only heuristic', 'weighted_kappa'].iloc[0]:.2f}), but still remained below the model range of {format_range(qualitative_agreement.loc[qualitative_agreement['kind'] == 'model', 'weighted_kappa'])}."
    )

summary_lines.extend(
    [
        "- These results support wording such as `recover`, `coarsely reproduce`, or `generate LR outputs with low average bias but wide dispersion`, rather than item-level interchangeability or close substitution.",
        "",
        "## Reliability-zone response",
        "",
        (
            "We added a central-versus-extreme reliability-zone analysis based on the calibration curves. "
            "The central zone was defined a priori as 0.2 <= LRReported <= 5.0; the extreme zone included LRReported < 0.2 or > 5.0. "
            f"All three models were summarized. For GPT-5, the central zone included {int(gpt5_central_reliability['n'])} rows and the extreme zone included {int(gpt5_extreme_reliability['n'])} rows. "
            f"GPT-5 median absolute log error increased from {gpt5_central_reliability['median_abs_log_error']:.2f} in the central zone to {gpt5_extreme_reliability['median_abs_log_error']:.2f} in the extreme zone, and RMSE_log increased from {gpt5_central_reliability['rmse_log']:.2f} to {gpt5_extreme_reliability['rmse_log']:.2f}. "
            f"The GPT-5 calibration slope was {gpt5_central_reliability['slope']:.2f} in the central zone and {gpt5_extreme_reliability['slope']:.2f} in the extreme zone, consistent with greater compression toward neutrality at the extremes. "
            "These findings support the manuscript wording that reliability was greatest in the central LR range and that extreme LRs should be interpreted with particular caution."
        ),
        "",
        "### Reliability-zone methods addition",
        "",
        (
            "As an additional requested sensitivity analysis, we defined reliability zones using the literature-reported LR. "
            "The central zone was defined as 0.2 <= LRReported <= 5.0, and the extreme zone was defined as LRReported < 0.2 or LRReported > 5.0. "
            "Within each zone, we repeated the same log-scale calibration and error summaries used elsewhere in the supplement, including calibration slope, median absolute log error, RMSE_log, typical multiplicative error factor, and Bland-Altman mean bias and 95% limits of agreement on the reported/model ratio scale."
        ),
        "",
        "### Reliability-zone results addition",
        "",
        (
            f"Reliability was greatest in the central LR range. Across models, median absolute log error was {format_range(central_reliability['median_abs_log_error'])} in the central zone versus {format_range(extreme_reliability['median_abs_log_error'])} in the extreme zone, and RMSE_log was {format_range(central_reliability['rmse_log'])} versus {format_range(extreme_reliability['rmse_log'])}. "
            f"Calibration slopes ranged from {format_range(central_reliability['slope'])} centrally and from {format_range(extreme_reliability['slope'])} at the extremes. "
            "These zone summaries were consistent with the calibration curves: model-generated values tracked reported LRs better near the center of the LR distribution, while extreme values showed greater error and compression toward neutrality."
        ),
        "",
        "## Laboratory discrepancy response",
        "",
        (
            "We added a descriptive GPT-5 laboratory discrepancy table using only the curated finding text. "
            "Because primary-study abstraction was outside scope, the table does not assign causal failure categories; instead, it flags whether the finding text contains an explicit numeric threshold or comparator, whether units are present, and whether the literature-reported LR is extreme. "
            f"Among the top 10 GPT-5 lab-like test-result discrepancies, {lab_threshold_count}/10 contained explicit numeric thresholds or comparators, {lab_unit_count}/10 contained units, and {lab_extreme_count}/10 had extreme reported LRs (<0.2 or >5.0). "
            "These descriptive findings support the limitation that laboratory LRs may require assay- and threshold-specific information that is not always recoverable from simplified finding text alone."
        ),
        "",
        "## Literal LR+ vs LR− sensitivity analysis",
        "",
        f"- All 700 rows classified successfully into source-derived true-direction labels using the original scrape prefixes: LR+ n={lr_plus_n} and LR− n={lr_minus_n}.",
        f"- This true LR+ / LR− partition was materially different from the LR < 1 / LR > 1 split. Among LR+ entries, {lr_plus_lt1} had LR < 1 and {lr_plus_eq1} had LR = 1; among LR− entries, {lr_minus_gt1} had LR > 1 and {lr_minus_eq1} had LR = 1.",
        f"- Positive association persisted within both true-direction strata. For LR+, Pearson r ranged from {format_range(model_true_direction.loc[model_true_direction['stratum'] == 'LR+', 'pearson_r'])}; for LR−, Pearson r ranged from {format_range(model_true_direction.loc[model_true_direction['stratum'] == 'LR−', 'pearson_r'])}.",
        f"- The true-direction Bland–Altman analysis now reports explicit mean bias and 95% limits of agreement on the reported/model ratio scale. Within LR+, {lr_plus_ba_details}. Within LR−, {lr_minus_ba_details}.",
        f"- The source-label-only LR+/LR− heuristic showed essentially no within-stratum association by construction: Pearson r = {label_only_lr_plus['pearson_r']:.2f} for LR+ and {label_only_lr_minus['pearson_r']:.2f} for LR−, whereas model correlations remained positive in both strata.",
        f"- Compared with the source-label-only heuristic overall, the models had lower RMSE_log ({format_range(model_overall['rmse_log'])} vs {label_only_overall['rmse_log']:.2f}) and higher R^2 ({format_range(model_overall['r2'])} vs {label_only_overall['r2']:.2f}).",
        "",
        "### Response letter",
        "",
        (
            "A requested analysis tested whether the apparent agreement could be driven by mixing literal LR+ and LR− entries. "
            "To answer that exact concern, we added a new stratified analysis using the original positive-finding versus negative-finding headings encoded during scraping rather than stratifying by whether LRReported was numerically above or below 1. "
            "Each row was classified as LR+ if the source table contributed it under a positive-finding heading and as LR− if it came from a negative-finding heading; before classification, two obvious scrape-prefix artifacts (`Patient  has:` and `Patient does has:`) were normalized. "
            f"All 700 rows classified successfully (LR+ n={lr_plus_n}; LR− n={lr_minus_n}). This true LR+ / LR− partition was not equivalent to the odds-direction split: {lr_plus_lt1} LR+ rows had LRReported < 1, {lr_plus_eq1} had LRReported = 1, {lr_minus_gt1} LR− rows had LRReported > 1, and {lr_minus_eq1} had LRReported = 1. "
            "Despite that stricter stratification, model association with literature-reported ln(LR) remained positive within both true LR+ and true LR− subsets, whereas a source-label-only LR+/LR− heuristic showed essentially no within-stratum association by construction. "
            f"Bland–Altman mean bias and 95% limits of agreement on the reported/model ratio scale were also reported within each true-direction stratum (LR+: {lr_plus_ba_details}; LR−: {lr_minus_ba_details}). "
            "We therefore believe this new analysis directly addresses the shortcut concern, while the prior LR < 1 versus LR > 1 and heuristic-baseline analyses remain complementary sensitivity analyses."
        ),
        "",
        "### Methods addition",
        "",
        (
            "As a requested supplementary analysis, we reconstructed true LR+ versus LR− labels from the source scraping convention used to build the workbook. "
            "Findings extracted under positive-finding headings were prefixed `Patient has:`, and findings extracted under negative-finding headings were prefixed `Patient does not have:` after removal of a leading `No` where applicable. "
            "Before classification, two obvious scrape-prefix artifacts (`Patient  has:` and `Patient does has:`) were normalized and repeated whitespace was collapsed. "
            "Rows were then stratified by the resulting source label (LR+ vs LR−), not by whether the reported LR was numerically greater than or less than 1. "
            "Within each true-direction subset, we repeated the same log-scale calibration, association, and error analyses used elsewhere in the supplement and additionally computed Bland–Altman mean bias and 95% limits of agreement on the reported/model ratio scale. "
            "We also evaluated a source-label-only heuristic that assigned the geometric-mean reported LR within the LR+ and LR− groups, with a leave-one-out sensitivity variant."
        ),
        "",
        "### Results addition",
        "",
        (
            f"All 700 finding-condition pairs were classified into source-derived true-direction labels (LR+ n={lr_plus_n}; LR− n={lr_minus_n}). "
            f"This partition differed materially from the LRReported < 1 versus LRReported > 1 split: among LR+ entries, {lr_plus_lt1} ({lr_plus_lt1_pct:.1%}) had LRReported < 1 and {lr_plus_eq1} ({lr_plus_eq1_pct:.1%}) had LRReported = 1, whereas among LR− entries, {lr_minus_gt1} ({lr_minus_gt1_pct:.1%}) had LRReported > 1 and {lr_minus_eq1} ({lr_minus_eq1_pct:.1%}) had LRReported = 1. "
            f"Nevertheless, positive association between literature-reported and model-generated ln(LR) persisted within both true-direction strata: within LR+, Pearson r ranged from {format_range(model_true_direction.loc[model_true_direction['stratum'] == 'LR+', 'pearson_r'])}; within LR−, Pearson r ranged from {format_range(model_true_direction.loc[model_true_direction['stratum'] == 'LR−', 'pearson_r'])}. "
            f"Bland–Altman mean bias and 95% limits of agreement on the reported/model ratio scale were as follows: within LR+, {lr_plus_ba_details}; within LR−, {lr_minus_ba_details}. "
            f"By contrast, the source-label-only LR+/LR− heuristic showed essentially no within-stratum association by construction (Pearson r = {label_only_lr_plus['pearson_r']:.2f} for LR+ and {label_only_lr_minus['pearson_r']:.2f} for LR−), supporting the conclusion that model performance reflects more than recovery of the coarse positive-versus-negative source label alone."
        ),
        "",
        "## Key output files",
        "",
        "- `Supp table - true LRplus vs LRminus cross-tab.csv`",
        "- `Supp fig - true LRplus vs LRminus calibration 2x3.pdf`",
        "- `Supp fig - true LRplus LRminus bland altman 2x3.pdf`",
        "- `Supp table - true LRplus vs LRminus calibration and association.csv`",
        "- `Supp table - LRplus LRminus label-only heuristic.csv`",
        "- `Supp fig - overall calibration 3 panel.pdf`",
        "- `Supp table - overall calibration and association.csv`",
        "- `Supp fig - within-stratum calibration 2x3.pdf`",
        "- `Supp table - within-stratum calibration and association.csv`",
        "- `Supp table - reliability zones central vs extreme.csv`",
        "- `Supp table - GPT5 laboratory discrepancies.csv`",
        "- `Supp fig - sign-only baseline calibration.pdf`",
        "- `Supp table - baseline performance.csv`",
        "- `Supp table - model vs heuristic comparison overall.csv`",
        "- `Supp table - model vs heuristic comparison by stratum.csv`",
        "- `Supp table - qualitative tier agreement vs baseline.csv`",
        "- `Supp table - posterior impact sensitivity.csv`",
        "- `Supp fig - posterior impact by pretest probability.pdf`",
    ]
)

summary_path = FOLLOWUP_DIR / "additional_requested_analyses_summary.md"
summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print(f"Saved summary -> {summary_path}")
print(summary_path.read_text(encoding='utf-8'))


## 11. Supplementary DOCX export



In [ ]:
from pathlib import Path
import shutil
import subprocess

try:
    from docx import Document
    from docx.enum.section import WD_ORIENT
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.shared import Inches
except Exception as exc:
    raise ImportError(
        "python-docx is required for follow-up table export. "
        "Install it in the project environment before running this notebook."
    ) from exc


def _load_table_for_docx(
    preferred_names: list[str],
    csv_filename: str,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    for name in preferred_names:
        value = globals().get(name)
        if isinstance(value, pd.DataFrame):
            df_value = value.copy()
            if columns is not None:
                missing = [column for column in columns if column not in df_value.columns]
                if missing:
                    raise KeyError(
                        f"Missing expected columns for DOCX export from in-memory table {name}: {missing}"
                    )
                return df_value.loc[:, columns].copy()
            return df_value
    csv_path = FOLLOWUP_DIR / csv_filename
    if not csv_path.exists():
        raise FileNotFoundError(f"Could not find table for DOCX export: {csv_path}")
    df_value = pd.read_csv(csv_path)
    if columns is not None:
        missing = [column for column in columns if column not in df_value.columns]
        if missing:
            raise KeyError(
                f"Missing expected columns for DOCX export from CSV {csv_filename}: {missing}"
            )
        return df_value.loc[:, columns].copy()
    return df_value


def _format_docx_value(value, column_name: str) -> str:
    if pd.isna(value):
        return "NA"
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    if isinstance(value, (np.floating, float)):
        value = float(value)
        if value.is_integer() and not any(
            token in column_name.lower() for token in ["p", "rho", "r2", "ccc", "slope", "error", "factor", "intercept", "bias", "loa"]
        ):
            return str(int(value))
        if "p" in column_name.lower() and value != 0 and abs(value) < 0.001:
            return f"{value:.2e}"
        return f"{value:.3f}"
    return str(value)


def _append_dataframe_table(
    document: Document,
    df_table: pd.DataFrame,
) -> None:
    table = document.add_table(rows=1 + len(df_table), cols=len(df_table.columns))
    table.style = "Table Grid"
    table.autofit = True

    for col_idx, column_name in enumerate(df_table.columns):
        table.cell(0, col_idx).text = str(column_name)

    for row_idx, (_, row) in enumerate(df_table.iterrows(), start=1):
        for col_idx, column_name in enumerate(df_table.columns):
            table.cell(row_idx, col_idx).text = _format_docx_value(row[column_name], str(column_name))


def _resolve_figure_path(figure_filename: str, optional: bool = False) -> Path | None:
    figure_path = FOLLOWUP_DIR / figure_filename
    if figure_path.exists():
        return figure_path
    if optional:
        return None
    raise FileNotFoundError(f"Required figure for DOCX export is missing: {figure_path}")


def _pdf_single_page(pdf_path: Path) -> bool:
    pdfinfo_path = shutil.which("pdfinfo")
    if pdfinfo_path is None:
        raise RuntimeError("pdfinfo is required to verify figure PDFs before DOCX export.")
    result = subprocess.run(
        [pdfinfo_path, str(pdf_path)],
        check=True,
        capture_output=True,
        text=True,
    )
    for line in result.stdout.splitlines():
        if line.startswith("Pages:"):
            pages = int(line.split(":", 1)[1].strip())
            return pages == 1
    raise RuntimeError(f"Could not determine PDF page count for {pdf_path}")


def _rasterize_figure_pdf(
    pdf_path: Path,
    temp_dir: Path,
) -> Path:
    pdftoppm_path = shutil.which("pdftoppm")
    if pdftoppm_path is None:
        raise RuntimeError("pdftoppm is required to insert PDF figures into the DOCX export.")
    if not _pdf_single_page(pdf_path):
        raise ValueError(f"Figure PDF must be single-page for DOCX export: {pdf_path}")

    png_base = temp_dir / pdf_path.stem
    subprocess.run(
        [pdftoppm_path, "-png", "-singlefile", str(pdf_path), str(png_base)],
        check=True,
        capture_output=True,
        text=True,
    )
    png_path = png_base.with_suffix(".png")
    if not png_path.exists():
        raise FileNotFoundError(f"Rasterized PNG was not created for {pdf_path}")
    return png_path


def _append_figure(
    document: Document,
    image_path: Path,
    width_emu: int,
) -> None:
    paragraph = document.add_paragraph()
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = paragraph.add_run()
    run.add_picture(str(image_path), width=width_emu)


DOCX_SECTION_SPECS = [
    {
        "heading": "Overall calibration and association",
        "subtitle": "Supp table - overall calibration and association.csv",
        "preferred_names": ["overall_metrics"],
        "csv_filename": "Supp table - overall calibration and association.csv",
        "columns": None,
        "methods_what": (
            "Calibration and association were evaluated on the natural-log likelihood-ratio scale for each model by comparing model-generated ln(LR) values against literature-reported ln(LR) values across all rows with complete paired data. The section reports calibration intercepts and slopes with Wald confidence intervals, R-squared, Pearson and Spearman association, log-scale error summaries, and Lin's concordance correlation coefficient as a secondary agreement metric."
        ),
        "methods_why": (
            "These analyses quantify whether model-generated LR outputs track the magnitude of published diagnostic evidence overall, rather than only reproducing the broad separation between negative and positive LRs. The combination of calibration, association, and dispersion metrics supports wording about recovery of signal while making clear whether outputs remain too dispersed for item-level interchangeability."
        ),
        "figures": [
            {"filename": "Supp fig - overall calibration 3 panel.pdf", "optional": False},
        ],
    },
    {
        "heading": "Within-stratum calibration and association",
        "subtitle": "Supp table - within-stratum calibration and association.csv",
        "preferred_names": ["within_metrics"],
        "csv_filename": "Supp table - within-stratum calibration and association.csv",
        "columns": None,
        "methods_what": (
            "The same log-scale calibration, association, and error analyses were repeated separately within the literature-reported LR < 1 and LR > 1 strata. This section therefore isolates performance among negative-evidence and positive-evidence items after removing the pooled between-stratum separation."
        ),
        "methods_why": (
            "This stratified analysis addresses the concern that an overall association could arise trivially from distinguishing only whether evidence points below or above neutrality. Persisting positive association within each stratum indicates that the models retain graded information beyond a simple directional split."
        ),
        "figures": [
            {"filename": "Supp fig - within-stratum calibration 2x3.pdf", "optional": False},
        ],
    },
    {
        "heading": "Reliability zones by reported LR range",
        "subtitle": "Supp table - reliability zones central vs extreme.csv",
        "preferred_names": ["reliability_zone_metrics"],
        "csv_filename": "Supp table - reliability zones central vs extreme.csv",
        "columns": [
            "zone",
            "reported_lr_range",
            "method",
            "n",
            "slope",
            "slope_ci_low",
            "slope_ci_high",
            "r2",
            "median_abs_log_error",
            "rmse_log",
            "typical_error_factor",
            "mean_bias_xfold",
            "loa95_lower_xfold",
            "loa95_upper_xfold",
            "interpretation",
        ],
        "methods_what": (
            "Reliability zones were defined from literature-reported LRs before examining model errors: a central zone of 0.2 <= LRReported <= 5.0 and an extreme zone of LRReported < 0.2 or LRReported > 5.0. Within each zone, the same log-scale calibration, error, and Bland-Altman metrics used elsewhere in the supplement were recomputed for each model."
        ),
        "methods_why": (
            "This analysis translates the calibration-curve observation into a response-ready reliability description. It identifies the LR range where model-generated outputs are most reliable and makes explicit that extreme LRs show larger errors and should be interpreted with greater caution."
        ),
        "figures": [],
    },
    {
        "heading": "Descriptive GPT-5 laboratory discrepancy table",
        "subtitle": "Supp table - GPT5 laboratory discrepancies.csv",
        "preferred_names": ["laboratory_discrepancy_top10"],
        "csv_filename": "Supp table - GPT5 laboratory discrepancies.csv",
        "columns": [
            "rank",
            "condition",
            "finding",
            "feature_type",
            "lr_reported",
            "lr_gpt5",
            "abs_log_error",
            "threshold_present",
            "unit_present",
            "extreme_reported_lr",
            "notes",
        ],
        "methods_what": (
            "The ten largest GPT-5 discrepancies among lab-like test-result findings were ranked by absolute log error. Candidate rows were defined from existing feature-type labels containing `test` plus laboratory, assay, or unit keywords in the curated finding text. The table flags whether the curated finding text includes an explicit numeric threshold or comparator, whether units are present, and whether the literature-reported LR is extreme (<0.2 or >5.0)."
        ),
        "methods_why": (
            "This descriptive table directly addresses the requested concern about laboratory discrepancies while avoiding unsupported causal taxonomy. It uses only the curated finding text and does not infer assay type, patient spectrum, reference standard, or study-specific context from primary diagnostic-accuracy studies."
        ),
        "figures": [],
    },
    {
        "heading": "True LR+ versus LR− reconstruction QC",
        "subtitle": "Supp table - true LRplus vs LRminus cross-tab.csv",
        "preferred_names": ["source_label_qc_export"],
        "csv_filename": "Supp table - true LRplus vs LRminus cross-tab.csv",
        "columns": [
            "source_lr_label",
            "n_total",
            "n_lr_lt_1",
            "n_lr_eq_1",
            "n_lr_gt_1",
            "pct_lr_lt_1",
            "pct_lr_eq_1",
            "pct_lr_gt_1",
        ],
        "methods_what": (
            "This quality-control table cross-tabulates the source-derived true LR+ versus LR− labels against the numerical direction of the reported LR (<1, =1, >1). The true-direction labels were reconstructed from the original scrape prefixes applied to positive-finding and negative-finding headings."
        ),
        "methods_why": (
            "This table is the direct tabular evidence that true LR+ / LR− labels are not equivalent to the odds-direction split. Non-zero mismatch cells show why LRReported < 1 versus > 1 cannot substitute for a genuine LR+ / LR− stratification."
        ),
        "figures": [],
    },
    {
        "heading": "True LR+ versus LR− stratified calibration, association, and Bland–Altman agreement",
        "subtitle": "Supp table - true LRplus vs LRminus calibration and association.csv",
        "preferred_names": ["true_label_metrics"],
        "csv_filename": "Supp table - true LRplus vs LRminus calibration and association.csv",
        "columns": None,
        "methods_what": (
            "The same log-scale calibration, association, and error analyses were repeated after stratifying rows by the source-derived true LR+ versus LR− labels rather than by whether the reported LR was numerically above or below 1. The resulting table now also reports Bland–Altman mean bias and 95% limits of agreement on the reported/model ratio scale, and the paired figures show both calibration and Bland–Altman agreement within each true-direction stratum."
        ),
        "methods_why": (
            "This section addresses the requested true LR+ versus LR− sensitivity analysis. Persisting positive association within both true LR+ and true LR− subsets argues against the explanation that model performance reflects only recognition of whether a row came from a positive-finding versus negative-finding source heading."
        ),
        "figures": [
            {"filename": "Supp fig - true LRplus vs LRminus calibration 2x3.pdf", "optional": False},
            {"filename": "Supp fig - true LRplus LRminus bland altman 2x3.pdf", "optional": False},
        ],
    },
    {
        "heading": "True LR+ versus LR− label-only heuristic comparison",
        "subtitle": "Supp table - LRplus LRminus label-only heuristic.csv",
        "preferred_names": ["source_label_comparison"],
        "csv_filename": "Supp table - LRplus LRminus label-only heuristic.csv",
        "columns": [
            "stratum",
            "method",
            "kind",
            "prediction_variant",
            "n",
            "slope",
            "slope_ci_low",
            "slope_ci_high",
            "r2",
            "pearson_r",
            "spearman_rho",
            "median_abs_log_error",
            "rmse_log",
            "typical_error_factor",
            "mean_bias_xfold",
            "loa95_lower_xfold",
            "loa95_upper_xfold",
            "ccc_log",
        ],
        "methods_what": (
            "This comparison table places the models alongside the constant-form source-label-only LR+/LR− heuristic that assigns the geometric-mean reported LR within the LR+ and LR− groups. A leave-one-out sensitivity variant was computed in the notebook for completeness but omitted from this compact response-ready table because it induces mechanical row-level dependence within each true-direction stratum. Results are shown overall and within each true-direction stratum using the same continuous-performance metrics as the main comparison tables, including explicit Bland–Altman mean bias and 95% limits of agreement on the reported/model ratio scale."
        ),
        "methods_why": (
            "This is the request-matched shortcut comparator: it tests how far one can get by knowing only whether a row originated from a positive-finding or negative-finding heading. The within-stratum rows demonstrate that once that coarse label is fixed, the heuristic carries essentially no graded information, whereas the models still do."
        ),
        "figures": [],
    },
    {
        "heading": "Baseline performance",
        "subtitle": "Supp table - baseline performance.csv",
        "preferred_names": ["baseline_metrics"],
        "csv_filename": "Supp table - baseline performance.csv",
        "columns": None,
        "methods_what": (
            "Two heuristic baselines were evaluated on the same ln(LR) scale: a sign-only heuristic assigning constant values within LR direction strata and a sign-plus-feature-type heuristic assigning constant values within sign-by-feature strata with sign-only fallback where groups were sparse. The table also retains leave-one-out sensitivity variants so the baseline results distinguish leakage-averse sensitivity checks from the pure constant heuristics used for the main shortcut comparison."
        ),
        "methods_why": (
            "These baselines operationalize the requested concern that models may be relying on a trivial shortcut, such as mapping positive findings to LR > 1 and negative findings to LR < 1 without recovering item-specific magnitude. Comparing the LLMs against these simple reference rules clarifies whether model performance exceeds what can be achieved by direction alone or by coarse sign-plus-feature grouping."
        ),
        "figures": [
            {"filename": "Supp fig - sign-only baseline calibration.pdf", "optional": False},
            {"filename": "Supp fig - sign-plus-feature baseline calibration.pdf", "optional": True},
        ],
    },
    {
        "heading": "Model vs heuristic comparison overall",
        "subtitle": "Supp table - model vs heuristic comparison overall.csv",
        "preferred_names": ["model_vs_heuristic_overall"],
        "csv_filename": "Supp table - model vs heuristic comparison overall.csv",
        "columns": [
            "method",
            "kind",
            "n",
            "slope",
            "slope_ci_low",
            "slope_ci_high",
            "r2",
            "pearson_r",
            "spearman_rho",
            "median_abs_log_error",
            "rmse_log",
            "typical_error_factor",
            "ccc_log",
        ],
        "methods_what": (
            "Overall model and heuristic results were consolidated into a single comparison table using common continuous-performance metrics, including calibration slope, R-squared, rank and linear association, median absolute log error, RMSE on the log scale, exponentiated typical error factor, and Lin's concordance correlation coefficient. The table uses the constant-form heuristics as the primary shortcut comparators."
        ),
        "methods_why": (
            "This summary table is designed to answer the response-ready question of whether the LLMs outperform a trivial rule when all rows are considered together. Presenting models and heuristics side by side on the same metric set makes the magnitude of any improvement easy to report in the rebuttal and supplement."
        ),
        "figures": [],
    },
    {
        "heading": "Model vs heuristic comparison by stratum",
        "subtitle": "Supp table - model vs heuristic comparison by stratum.csv",
        "preferred_names": ["model_vs_heuristic_by_stratum"],
        "csv_filename": "Supp table - model vs heuristic comparison by stratum.csv",
        "columns": [
            "stratum",
            "method",
            "kind",
            "n",
            "slope",
            "slope_ci_low",
            "slope_ci_high",
            "r2",
            "pearson_r",
            "spearman_rho",
            "median_abs_log_error",
            "rmse_log",
            "typical_error_factor",
            "ccc_log",
        ],
        "methods_what": (
            "The same model-versus-heuristic comparison metrics were then reported separately for LR < 1 and LR > 1 items. This table preserves the stratum labels so that continuous performance can be compared directly within negative-evidence and positive-evidence subsets."
        ),
        "methods_why": (
            "A by-stratum comparison is needed because shortcut baselines can appear more competitive in pooled data if they only separate values around neutrality. Displaying model and heuristic performance within each sign stratum shows whether the LLMs continue to recover graded LR information after the broad directional distinction has already been fixed."
        ),
        "figures": [],
    },
    {
        "heading": "Qualitative tier agreement vs baseline",
        "subtitle": "Supp table - qualitative tier agreement vs baseline.csv",
        "preferred_names": ["qualitative_agreement"],
        "csv_filename": "Supp table - qualitative tier agreement vs baseline.csv",
        "columns": None,
        "methods_what": (
            "Reported and model-generated LRs were mapped into the seven ordered qualitative tiers used in the main analysis, and agreement was summarized with quadratic-weighted Cohen's kappa, exact-match rate, and within-one-tier agreement. The same qualitative mapping was applied to both model outputs and heuristic baselines."
        ),
        "methods_why": (
            "This ordinal analysis evaluates whether the models recover more than just the sign of the LR by testing whether they place items into the correct strength-of-evidence tier more often than simple heuristics do. It complements the continuous-scale analyses with a supplement-ready measure that is easier to interpret clinically."
        ),
        "figures": [],
    },
    {
        "heading": "Posterior impact sensitivity",
        "subtitle": "Supp table - posterior impact sensitivity.csv",
        "preferred_names": ["posterior_impact"],
        "csv_filename": "Supp table - posterior impact sensitivity.csv",
        "columns": None,
        "methods_what": (
            "For representative pretest probabilities, post-test probabilities were computed using literature-reported LRs and model-generated LRs, and absolute posterior differences were summarized by median, 90th percentile, and 95th percentile. Threshold-flip rates were also calculated at representative posterior-probability cut points to show how often model-generated posteriors would cross decision-relevant boundaries."
        ),
        "methods_why": (
            "This sensitivity analysis translates LR disagreement into a clinically interpretable scale and therefore helps contextualize the practical meaning of wide log-scale dispersion. It is especially useful for explaining why modest average bias can coexist with posterior changes that are too large to justify language implying close substitution."
        ),
        "figures": [
            {"filename": "Supp fig - posterior impact by pretest probability.pdf", "optional": False},
        ],
    },
]

doc = Document()
section = doc.sections[0]
section.orientation = WD_ORIENT.LANDSCAPE
section.page_width, section.page_height = section.page_height, section.page_width
section.top_margin = Inches(0.5)
section.bottom_margin = Inches(0.5)
section.left_margin = Inches(0.5)
section.right_margin = Inches(0.5)
usable_width = section.page_width - section.left_margin - section.right_margin

doc.add_heading("Supplementary Tables - Additional requested analyses", level=1)
doc.add_paragraph(
    "This document compiles the follow-up calibration, heuristic comparison, qualitative agreement, and posterior-impact tables and matched figures exported by this notebook."
)

tmp_doc_dir = WORKING_DIR / "tmp" / "docs"
tmp_doc_dir.mkdir(parents=True, exist_ok=True)
raster_paths: list[Path] = []
resolved_tables = []

try:
    for spec in DOCX_SECTION_SPECS:
        document_heading = doc.add_heading(spec["heading"], level=2)
        document_heading.alignment = WD_ALIGN_PARAGRAPH.LEFT
        doc.add_paragraph(spec["methods_what"])
        doc.add_paragraph(spec["methods_why"])

        for figure_spec in spec["figures"]:
            figure_path = _resolve_figure_path(
                figure_spec["filename"],
                optional=figure_spec.get("optional", False),
            )
            if figure_path is None:
                continue
            png_path = _rasterize_figure_pdf(figure_path, tmp_doc_dir)
            raster_paths.append(png_path)
            _append_figure(doc, png_path, usable_width)

        doc.add_paragraph(spec["subtitle"])
        resolved = _load_table_for_docx(
            spec["preferred_names"],
            spec["csv_filename"],
            columns=spec["columns"],
        )
        resolved_tables.append((spec, resolved))
        _append_dataframe_table(doc, resolved)
        doc.add_paragraph("")

    followup_docx_path = FOLLOWUP_DIR / "Supp Tables - additional requested analyses.docx"
    doc.save(followup_docx_path)
finally:
    for raster_path in raster_paths:
        if raster_path.exists():
            raster_path.unlink()
    if tmp_doc_dir.exists() and not any(tmp_doc_dir.iterdir()):
        tmp_doc_dir.rmdir()

print(f"Saved DOCX -> {followup_docx_path}")
for spec, resolved in resolved_tables:
    print(f"{spec['csv_filename']}: {resolved.shape[0]} rows x {resolved.shape[1]} columns")
